# Feature Research Notebook (PR 42)

Training-side research on the accepted feature representations of both cascades. Commit 2 built the
foundation (Cells 1–13): one labelled top-to-bottom path, the cascades' split drawn **training side only**, the
inventory of the accepted features, feature contracts and stage feature populations, and the frozen
references later commits compare against. Commit 3, Step 3A (Cells 14–21) audits the accepted target-derived
encodings: for a row being encoded, which rows and labels reach its value, and which of those labels could
have been known when it was contacted. Step 3B (Cells 22–30) builds past-only versions of the Stage 1
encodings, measures their predictive effect against the accepted ones on identical rows, records the
feature-contract decisions and designs the PR 49 temporal robustness protocol. Commit 4, Step 4A
(Cells 31–39) reproduces the PR 39 Stage 2 NSD/ECI bin evidence with fold-isolated features and diagnoses
the lift drift it left open. Step 4B (Cells 40–48) registers a comparison, then compares fold-fitted NSD/ECI
binning procedures with the frozen bins on identical held-out rows and tests `eci_warm` as a filter on partner
rules. Commit 5, Step 5A (Cells 49–59) refits the frozen Stage 3 EBM on the training side, compares four
registered simplifications with it on identical out-of-fold rows, plots every EBM shape across folds and contact
periods, and measures complementarity with Stage 1 where the accepted artifacts are available. Step 5B
(Cells 60–68) makes the EBM's interaction selection procedural and fitted inside every training fit, checks whether
it recovers the three accepted pairs, and reselects the pairs of the two carried-forward candidates. Each run writes
seven write-once research records. Commit 6, Step 6A adds no cell and no record: it consolidates the accepted
records into feature-contract recommendations and the handoff to PRs 43–49 (the conclusion at the end).

**Rules** (`tests/notebooks/test_notebook_contracts.py`, G5 and G7)

- Training rows only. Cell 2 draws the cascades' split inside `draw_training_side`, which returns the
  training side. No test row, test label, test row identity, test prediction or test metric is bound
  in this notebook.
- Accepted artifacts are opened only through the PR 38 boundary extractors (Cell 11).
- Nothing here saves a model or a cascade artifact, edits production feature code or writes any file except
  this run's records, `research_logs/pr42/<run_id>/feature_inventory.json`, `encoding_audit.json`,
  `temporal_validity.json`, `binning_drift.json`, `binning_candidates.json`, `stage3_simplification.json` and
  `interaction_selection.json` (each with its `.sha256`).
  Step 3B fits Stage 1 research models in memory, on training rows, with the accepted PR 39 settings; none is
  kept. Steps 4A and 4B fit only through the PR 39 and Commit 3 functions they reuse, in memory. Steps 5A and 5B fit
  Stage 3 EBMs in memory through the shared Stage 3 code with the accepted settings; none is kept.
- Run top to bottom in a fresh kernel, from `feature_research/` or `bank_pipeline/`.

**Sources** (`documentation/notebook_map.md`, section 2.1): **E** = `exploratory_feature_research.ipynb`,
**G** = `glass_cascade.ipynb`, **B** = `blackbox_cascade.ipynb`. Review documents:
`research_logs/pr42/feature_inventory.md` (Commit 2), `research_logs/pr42/encoding_audit.md` (Step 3A),
`research_logs/pr42/temporal_validity.md` (Step 3B), `research_logs/pr42/binning_drift.md` (Step 4A) and
`research_logs/pr42/binning_candidates.md` (Step 4B), `research_logs/pr42/stage3_simplification.md` (Step 5A),
`research_logs/pr42/interaction_selection.md` (Step 5B) and `research_logs/pr42/feature_contract_recommendations.md`
(Step 6A: the recommendations and handoff).

## Cell 1 — Imports and path setup
From E 1 and E 3: puts `bank_pipeline/` on `sys.path` (and removes the notebook's own folder, so sibling
modules cannot be imported by bare name), applies the shared seed and plot settings, and imports the
PR 42 research modules.

In [ ]:
# =============================================================================
# CELL 1: PATH BOOTSTRAP, IMPORTS & SETUP
# =============================================================================
import sys
from pathlib import Path

import pandas as pd

_here = Path.cwd().resolve()

BANK_ROOT = next(
    (p for p in (_here, *_here.parents)
     if (p / "feature_research" / "__init__.py").is_file() and (p / "shared").is_dir()),
    None,
)
if BANK_ROOT is None:
    raise RuntimeError(f"Could not locate bank_pipeline/ above {_here}")

def _is_here(p):
    try:
        return Path(p or ".").resolve() == _here
    except OSError:
        return False

sys.path[:] = [p for p in sys.path if not _is_here(p)]
if str(BANK_ROOT) not in sys.path:
    sys.path.insert(0, str(BANK_ROOT))

from feature_research import RANDOM_SEED, apply_global_settings
from feature_research import binning_candidates as BC
from feature_research import binning_drift as BD
from feature_research import encoding_audit as EA
from feature_research import feature_inventory as FI
from feature_research import frozen_references as FR
from feature_research import research_records as RR
from feature_research import interaction_selection as IS
from feature_research import stage3_simplification as SS
from feature_research import temporal_validity as TV

apply_global_settings()
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 220)
print(f"bank_pipeline root: {BANK_ROOT}")

## Cell 2 — Training side of the cascades' split
Replaces E 2, E 4 and E 5. `draw_training_side()` runs the protocol every notebook runs —
`build_database()` → `BankPreprocessor(drop_leaky=True).fit_transform()` → `sanity_check_data` →
`GlobalSplitManager(test_size=0.20, random_state=42).create_split(..., target_col="y")` — inside one call,
and returns only the training rows, their labels, the train-side identities and the PR 38 `TrainReference`
(fingerprints only). The population, the test split and the split manager never leave the call, and the
protocol's own console output (full-population and test-split counts) is discarded.

In [ ]:
# ============================================================
# CELL 2: TRAINING SIDE OF THE CASCADES' SPLIT
# ============================================================
from feature_research.training_side import draw_training_side

TRAIN = draw_training_side()

## Cell 3 — Run identity
A new run id (`c5_<YYYYMMDD_HHMMSS>`; Commit 4 runs were `c4_`), every seed by name, the code identity (HEAD, uncommitted tracked
changes, LF SHA-256 of the files the inventory and the audit trace, whole-tree digest) and the environment. A
run id is used once: Cell 13 creates the run folder and refuses an existing one, and Cells 21, 30, 39, 48,
59 and 68 add the run's second to seventh records beside the first. No record is ever written over.

In [ ]:
# ============================================================
# CELL 3: RUN IDENTITY — run id, seeds, code identity
# ============================================================
SEEDS = {"split": RANDOM_SEED, "fold_plans": RANDOM_SEED,
         "target_probe": FI.PROBE_SEED, "parity_folds": RANDOM_SEED,
         "encoding_audit": EA.SEED, "temporal_validity": TV.SEED,
         "binning_drift": BD.SEED, "binning_candidates": BC.SEED,
         "stage3_simplification": SS.SEED, "interaction_selection": IS.SEED}
RUN = RR.start_run(commit=5, seeds=SEEDS)
print(RUN.describe())

## Cell 4 — Feature types
From E 6, on the training rows. `classify_features` types an integer column with at most 10 distinct values as
categorical and every other column as numeric, so `job` (12 codes) is numeric and `previous` categorical — as in
E 6's own output, although E 6's markdown table lists those two the other way round.

In [ ]:
# ============================================================
# CELL 4: FEATURE TYPING
# ============================================================
from feature_research import classify_features

NUMERIC_FEATURES, CATEGORICAL_FEATURES = classify_features(TRAIN.frame(), TRAIN.target_col)

## Cell 5 — Stage feature populations
What each stage of each arm consumes, each population checked against the production registry it names
(`LR_FEATURES`, the MLP's `STAGE1_FEATURES`, `RF_SOURCE_FEATURES`, `RF_FEATURES_BINARY`, `EBM_FEATURES`,
`define_ebm_interactions`, `STAGE4_FEATURES`) and against the cascade notebook cells that build or read it
(G 5, G 8, G 8B, G 17–19; B 7, B 9, B 16–18, B 30–31, B 35–36; source only, nothing executed). The research
registries in `feature_research.feature_engineering` are checked too: they are not what the cascades consume.

In [ ]:
# ============================================================
# CELL 5: STAGE FEATURE POPULATIONS — registries and call sites
# ============================================================
POPULATIONS = FI.population_table()
REGISTRY_CHECKS = FI.check_registries()
CALL_SITE_CHECKS = FI.check_call_sites()
print(FI.require(REGISTRY_CHECKS, "registries"))
print(FI.require(CALL_SITE_CHECKS, "call sites"))
POPULATIONS[["key", "model", "n_features", "registry", "contract", "cells"]]

## Cell 6 — Per-feature inventory
One row per (stage, arm, consumed feature): kind, definition, source variables (raw columns, transitively),
target dependence and the fitted features it comes from, fold refits, and the feature's own test-time rule —
stated from the production code, not by reading the test split. The final-fit and cross-validation scopes, and
where they differ, are the population's (`POPULATIONS["fit"]`, Cell 5). Intermediate DAG nodes are in `NODES`.

In [ ]:
# ============================================================
# CELL 6: PER-FEATURE INVENTORY
# ============================================================
INVENTORY = FI.feature_table()
NODES = FI.node_table()
print(INVENTORY.groupby(["stage", "arm"]).size().rename("features").to_string())
INVENTORY[["stage", "arm", "feature", "kind", "target_dependence", "source_variables", "out_of_fit_rule"]]

## Cell 7 — Feature builds on the training rows
From G 5 / B 7 (Stage 1), B 16–17 (Stage 2 sources and bins) and E 11–13 (the feature DAG and its validation),
with the **production** engineers: each fitted on all training rows and applied to them, as the final fits do.
Checks: every declared node is produced by each implementation that claims it, the training-path bins equal the
test-path binning applied to the same rows, every bin family is mutually exclusive and complete, and every
consumed value is finite. `check_nsd_reference` confirms how the density encodes its own fit rows: every production
NSD fit (the final fits and every Stage 2 and Stage 3 fold fit) has more than 10,000 rows, so only a seeded sample
of 5,000 reference rows contributes labels, and the full-train fits of both implementations keep exactly 5,000.

In [ ]:
# ============================================================
# CELL 7: FEATURE BUILDS ON THE TRAINING ROWS
# ============================================================
TRAINING_FRAMES = FI.build_training_frames(TRAIN.X_train, TRAIN.y_train)
DAG_CHECKS = FI.check_dag_columns(TRAINING_FRAMES)
NSD_CHECKS = FI.check_nsd_reference(TRAINING_FRAMES)
print(FI.require(DAG_CHECKS, "DAG columns"))
print(FI.require(NSD_CHECKS, "NSD reference sample"))
print(TRAINING_FRAMES["nsd_fit_sizes"].to_string(index=False))
BUILD_SUMMARY = FI.training_build_summary(TRAINING_FRAMES, TRAIN.y_train)
BUILD_SUMMARY

## Cell 8 — Target dependence, verified
Each implementation is fitted twice on the same training rows, with the true labels and with a seeded
permutation of them, and a fixed sample of those rows is transformed by both fits. A column whose values change
depends on the labels. Every declaration (`none`, `fitted`, `inherited`) must agree; a bin is judged through
its family or its source (a bin is a fixed-threshold function of its source).
This verifies dependence on labels only: whether that dependence is temporally valid is Commit 3.

In [ ]:
# ============================================================
# CELL 8: TARGET-DEPENDENCE PROBE (label permutation)
# ============================================================
PROBE = FI.probe_target_dependence(TRAIN.X_train, TRAIN.y_train)
print(FI.require_probe(PROBE))
PROBE.pivot_table(index=["implementation", "declared"], columns="observed", values="column",
                  aggfunc="count", fill_value=0)

## Cell 9 — Fitting and fold-refit settings
The settings the inventory states, read from the production code: Stage 1 per-fold encoder and scaler refits
and fold counts, the accepted Stage 2 run and its `fold_label_excluded` inner-fold pathway, and both Stage 3
configurations (`per_fold`, 5 tuning and 10 reporting folds, seed 42).

In [ ]:
# ============================================================
# CELL 9: FIT PROTOCOLS
# ============================================================
FIT_CHECKS = FI.check_fit_protocols()
print(FI.require(FIT_CHECKS, "fit protocols"))
FIT_CHECKS[["stage", "check", "passed", "detail"]]

## Cell 10 — Frozen references
Recomputed from this branch and compared with the accepted PR 39 records (read after their `.sha256`
sidecars): the Stage 1 feature contracts, the Stage 2 binning identity, the Stage 3 feature DAG and block-cache
identity, the three EBM interaction pairs, the Stage 4 inputs and the train-side population. `MISMATCH` stops
the notebook; `local verification` items are checked by `tests/realdata/test_pr42_frozen_references.py`.

In [ ]:
# ============================================================
# CELL 10: FROZEN REFERENCES
# ============================================================
FROZEN = pd.DataFrame(FR.frozen_references(TRAIN))
STAGE3_DAG = FR.dag_implementation()
if (FROZEN["status"] == "MISMATCH").any():
    raise AssertionError(FROZEN[FROZEN["status"] == "MISMATCH"].to_string())
print(FROZEN["status"].value_counts().to_string())
FROZEN[["name", "status", "computed", "recorded"]]

## Cell 11 — Accepted artifacts, through the PR 38 boundary extractors only
The frozen Phase 1 Stage 1–3 artifacts (PR 39 Commit 4 manifest), each verified by size and SHA-256 and read
by its boundary extractor, which reads allowlisted training-side fields only. Roles without an extractor (the
black-box MLP and XGBoost) are not opened, and fields the extractors do not expose (such as the EBM's stored
interaction pairs) are not read: Cell 10 states those from the production code. Artifacts are local (`models/` is Git-ignored): an absent file is
recorded as `not available locally`; a changed file or a failed extractor check stops the notebook.

In [ ]:
# ============================================================
# CELL 11: ACCEPTED ARTIFACTS THROUGH THE PR 38 BOUNDARY EXTRACTORS
# ============================================================
ACCEPTED = FR.accepted_artifact_views(TRAIN.train_reference)
ACCEPTED_TABLE = pd.DataFrame(ACCEPTED).reindex(
    columns=["role", "status", "extractor", "checks", "checks_passed", "path"])
if (ACCEPTED_TABLE["status"] == "FAILED").any():
    raise AssertionError(pd.DataFrame(ACCEPTED).to_string())
ACCEPTED_TABLE

## Cell 12 — Duplicated feature-engineering implementations
For the Commit 6 decision (notebook map section 5): the functions `shared/stage2/feature_engineering.py` copies
from `feature_research/feature_engineering/`, their current importers (AST of every Python file and the
version-controlled notebooks), and how far the two implementations — and Stage 1's mirrored features — agree on
the training rows, in-sample and on the shared 5 folds' held-out rows. Values are compared with `compare_values`:
a value missing on one side only counts as a difference, values missing on both sides match, and the maximum
difference is taken over pairs where both values are finite. Nothing is consolidated here.

In [ ]:
# ============================================================
# CELL 12: DUPLICATED FEATURE-ENGINEERING IMPLEMENTATIONS
# ============================================================
DUPLICATES = pd.DataFrame(FI.DUPLICATE_PAIRS)
CALLERS = FI.duplicate_callers()
PARITY, BINNING_PORT = FI.duplicate_parity(TRAIN.X_train, TRAIN.y_train)
print(FI.callers_summary(CALLERS))
print(BINNING_PORT)
PARITY[PARITY["rows_differ"] > 0]

## Cell 13 — Research record: feature inventory
The run's first record: writes `research_logs/pr42/<run_id>/feature_inventory.json` and its `.sha256` sidecar
once (creating the run folder), then reads it back through its sidecar and content digest. `content_sha256`
excludes the run id, times, code and environment, so a reproducing run has the same value. Its sections are the
Commit 2 inventory unchanged; Cell 21 compares them with the verified Commit 2 record.

In [ ]:
# ============================================================
# CELL 13: WRITE THE RESEARCH RECORD
# ============================================================
RECORD_PATH = RR.write_run_record(RUN, "feature_inventory", {
    "scope": "PR 42 Commit 2: feature research foundation and feature inventory (training side)",
    "training_side": {"protocol": dict(TRAIN.protocol), "identities": dict(TRAIN.identities)},
    "feature_typing": {"numeric": NUMERIC_FEATURES, "categorical": CATEGORICAL_FEATURES},
    "populations": POPULATIONS.to_dict("records"),
    "inventory": INVENTORY.to_dict("records"),
    "nodes": NODES.to_dict("records"),
    "checks": {"registries": REGISTRY_CHECKS.to_dict("records"),
               "call_sites": CALL_SITE_CHECKS.to_dict("records"),
               "dag_columns": DAG_CHECKS.to_dict("records"),
               "nsd_reference": NSD_CHECKS.to_dict("records"),
               "fit_protocols": FIT_CHECKS.to_dict("records")},
    "training_frames": BUILD_SUMMARY.to_dict("records"),
    "nsd_reference": {"fit_sizes": TRAINING_FRAMES["nsd_fit_sizes"].to_dict("records"),
                      "full_train": TRAINING_FRAMES["nsd_reference"].to_dict("records")},
    "target_probe": {"rows": FI.PROBE_ROWS, "results": PROBE.to_dict("records")},
    "frozen_references": FROZEN.to_dict("records"),
    "stage3_dag": STAGE3_DAG,
    "accepted_artifacts": ACCEPTED,
    "duplicates": {"pairs": list(FI.DUPLICATE_PAIRS), "related": list(FI.RELATED_COPIES),
                   "callers": CALLERS, "parity": PARITY.to_dict("records"),
                   "binning_port": BINNING_PORT},
})
RECORD = RR.read_run_record(RUN.run_id, "feature_inventory")
print(f"record  : {RECORD_PATH.relative_to(BANK_ROOT)}")
print(f"content : {RECORD['content_sha256']}")

## Cell 14 — The PR 42 time index
The rows of `bank-additional-full.csv` are in date order, May 2008 to November 2010, with no year and no day of
month. `verify_source_order` checks that each training row's index label is its line in the CSV — through the
database load (`raw_bank` rowids in insertion order, `transform.sql` reading `ORDER BY rowid`), preprocessing
and the split — reading only the training rows' lines. `build_time_index` then derives, from training rows only:
the **period** (year inferred where the month decreases in source order; checked against the monthly indicators
and the calendar) and the **contact day** (a run of rows sharing year, month, weekday and the daily
`euribor3m`). The finest defensible resolution is the contact day: contacts of the same day, and when any
outcome became known, are unresolved (`EA.RESOLUTION`).

In [ ]:
# ============================================================
# CELL 14: TIME INDEX — source order, periods and contact days
# ============================================================
SOURCE_CHECKS = EA.verify_source_order(TRAIN.X_train, TRAIN.y_train)
TIME_INDEX = EA.build_time_index(TRAIN.X_train)
TIME_CHECKS = EA.time_index_checks(TRAIN.X_train, TIME_INDEX)
PERIODS = EA.period_table(TIME_INDEX, TRAIN.y_train)
print(EA.require(SOURCE_CHECKS, "source order"))
print(EA.require(TIME_CHECKS, "time index"))
RECURRING_DAY_KEYS = EA.recurring_day_keys(TIME_INDEX, TRAIN.X_train)
print(f"{TIME_INDEX['day'].nunique()} contact days in {len(PERIODS)} periods; "
      f"{RECURRING_DAY_KEYS} day keys recur in a later week of their month")
PERIODS

## Cell 15 — How the encodings are built
`EA.CONSTRUCTION` states, from the production code, how each audited encoding is fitted, applied and smoothed,
its fallbacks, and how a fit row enters its own value. On the full training split: each Stage 1 euribor bin's
rows, rate and the periods it spans, and each day x month cell's rows and the years it pools (the key has no year).

In [ ]:
# ============================================================
# CELL 15: ENCODING CONSTRUCTION
# ============================================================
ENCODINGS = pd.DataFrame(EA.CONSTRUCTION).T
STAGE1_BUILD = EA.stage1_construction(TRAIN.X_train, TRAIN.y_train, TIME_INDEX)
BINS, CELLS = STAGE1_BUILD["bins"], STAGE1_BUILD["cells"]
print(STAGE1_BUILD["settings"].to_string(index=False))
print(f"euribor bins spanning at most three periods: {(BINS['periods'] <= 3).sum()}/{len(BINS)}; "
      f"day x month cells pooling more than one year: {(CELLS['n_years'] > 1).sum()}/{len(CELLS)}")
BINS

## Cell 16 — Fitting paths
Every path that fits an audited encoding, as training positions, from the production fold plans: Stage 1
(both arms) final fit, 10 outer folds (tuning and OOF) and 10 x 10 inner calibration folds; Stage 2 (both
routers) final fit, its 5 label-excluded inner folds, 5 outer folds and their 5 x 5 label-excluded inner folds;
Stage 3 (both arms) final fit, 5 tuning, 10 reporting and 5 x 5 inner folds. The checks confirm each plan
partitions what it covers and where plans coincide (so evidence is shared); outer fold 0 and its inner plan are
compared with what the accepted Stage 2 run recorded.

In [ ]:
# ============================================================
# CELL 16: FITTING PATHS
# ============================================================
PATHS = EA.fit_paths(TRAIN.y_train)
PATH_TABLE = EA.path_table(PATHS)
PLAN_CHECKS = EA.plan_checks(TRAIN.y_train, PATHS)
ROUTER_PLAN = EA.router_plan_reference(TRAIN.X_train, TRAIN.y_train, PATHS)
print(EA.require(PLAN_CHECKS, "fold plans"))
print(EA.require(ROUTER_PLAN.rename(columns={"item": "check"}), "accepted Stage 2 fold 0 plan"))
PATH_TABLE

## Cell 17 — Fold isolation
Label flips through the production fold builders, each paired with a control that must change (so the check
can fail): Stage 1 `build_fold_designs` / `inner_fold_designs` (the calls both arms make), the Stage 2
label-excluded encoder, and one training row's own label in the full-train fit (mechanism a, measured). The
Stage 2 outer and Stage 3 fold fits are checked at the call sites: each is passed the fold's training labels only.
That the accepted fits used these refits is read from their records: the boundary extractors' fold checks (Cell 11
here, and the verified Commit 2 record, `RR.COMMIT2_INVENTORY`) and the accepted Stage 1 fits of both arms (PR 39
Commit 2 evidence; the MLP has no extractor).

The black-box MLP early-stops on a subset drawn **inside** each fit, after that fit's encoders were fitted on all of
its rows (`EA.MLP_EARLY_STOPPING_ORDER`). The subset's labels therefore enter the encodings of both the optimization
rows and the validation rows; a properly isolated split would exclude the whole subset from encoder fitting
(`EA.MLP_ISOLATED_SPLIT`). `mlp_early_stopping` reproduces the split on every Stage 1 fit (confirmed against the
estimator's own validation ROC-AUC) and measures the subset's share of both sides' label mass. Effects on the stopping
epoch, predictions and AUC are not quantified. Outer held-out isolation is a separate finding: those rows are outside
the subset, the encoders and the network, and are assessed by the flips above (`EA.MLP_OUTER_ISOLATION`).

In [ ]:
# ============================================================
# CELL 17: FOLD ISOLATION — label flips and in-fold label arguments
# ============================================================
COMMIT2 = RR.read_run_record(*RR.COMMIT2_INVENTORY)
ISOLATION = EA.isolation_checks(TRAIN.X_train, TRAIN.y_train, PATHS)
ACCEPTED_FITS = EA.accepted_fit_scopes(ACCEPTED, COMMIT2)
EARLY_STOPPING = EA.mlp_early_stopping(TRAIN.X_train, TRAIN.y_train, PATHS)
print(EA.require(ISOLATION, "fold isolation"))
print(EA.require(ACCEPTED_FITS.rename(columns={"item": "check"}), "accepted fits use fold-fitted encoders"))
print(EA.require(EARLY_STOPPING["checks"], "MLP early-stopping split reproduced"))
print(EARLY_STOPPING["summary"].round(5).to_string(index=False))
print(EARLY_STOPPING["finding"])
ISOLATION[["check", "passed", "detail"]]

## Cell 18 — Which labels reach each encoded value, and when
Every encoded value on every path is written as a combination of fit-row labels (bin or cell members, the
smoothing prior, the kernel's reference rows, the curvature's neighbouring bins), reconstructed and compared with
the production output, and its label mass is split by when each contributing contact happened relative to the
encoded row: earlier day, same day (unresolved), later day of the same month, later month. Roles: `held_out`
(a fold's held-out rows, each located among the fit's source rows so that its own label would be measured: b, and
c), `fit_rows` (the fit's own rows, i.e. the training representation: a, and its look-ahead) and `final_mapping`
(the final fits at each training row's contact day, standing for a new row contacted then; by convention its own
label counts as a same-day contact). Every comparison with production fails on a nonfinite value, and the cell
stops unless the completed comparisons match the expected ones (`EA.COMPARISON_POLICY`: the Stage 2 fold fits' own
rows are the only intentional skips) and no held-out encoding carries its own label.

In [ ]:
# ============================================================
# CELL 18: ATTRIBUTION BY FITTING PATH AND CONTACT TIMING
# ============================================================
AUDIT = EA.audit_paths(TRAIN.X_train, TRAIN.y_train, TIME_INDEX, PATHS)
SUMMARY = EA.path_summary(AUDIT)
COVERAGE = EA.comparison_coverage(AUDIT, PATHS)
HELD_OUT_OWN = EA.held_out_own_labels(SUMMARY)
print(EA.require(COVERAGE, "reconstruction: expected versus completed comparisons"))
print(COVERAGE.iloc[-1]["check"], "->", COVERAGE.iloc[-1]["detail"])
print(COVERAGE.iloc[-2]["check"], "->", COVERAGE.iloc[-2]["detail"])
print(EA.require(HELD_OUT_OWN, "held-out own labels (mechanism b, measured)"))
print(f"largest |reconstructed - production| over passed comparisons: "
      f"{SUMMARY['max_abs_diff'].dropna().max():.1e} (tolerance {EA.TOLERANCE:.0e})")
SUMMARY[["feature", "stage", "path", "role", "fits", "rows", "self_rows", "later_month_rows",
         "any_later_rows", "later_share_mean", "later_share_p90", "majority_later_rows"]]

## Cell 19 — The final mappings by contact period
The final fits score new rows. At each training row's contact day and covariates, the mean share of the label mass
from later days of the same month and from later months, by period: what a contemporaneous new row's encoding is
built from under the random split.

In [ ]:
# ============================================================
# CELL 19: FINAL MAPPINGS BY PERIOD
# ============================================================
BY_PERIOD = EA.mapping_by_period(AUDIT)
BY_PERIOD.assign(later=BY_PERIOD["later_day_same_month"] + BY_PERIOD["later_month"]).pivot_table(
    index="period", columns=["stage", "feature"], values="later").round(3)

## Cell 20 — Provenance and findings
Where the constants and the frozen Stage 2 thresholds came from (read from the PR 39 Step 3A record), separately
from the labels a row's transformation uses, and the finding per feature, stage and fitting path: mechanisms
(a), (b) and (c) in counts with denominators (`FINDINGS`; `FAMILY_FINDINGS` per family). Interpretation, limits and evidence locations:
`research_logs/pr42/encoding_audit.md`.

In [ ]:
# ============================================================
# CELL 20: PROVENANCE AND FINDINGS
# ============================================================
PROVENANCE = EA.provenance_table()
FINDINGS = EA.findings_table(SUMMARY)
FAMILY_FINDINGS = EA.family_findings(SUMMARY)
print(PROVENANCE[["item", "provenance"]].to_string(index=False))
FINDINGS[["feature", "stage", "path", "fits", "a_self", "b_held_out", "c_rows", "c_later_month",
          "c_any_later_day", "c_later_share_mean", "c_later_majority", "same_day_share_mean"]]

## Cell 21 — Research record: encoding audit
The run's second record, written beside the first: `research_logs/pr42/<run_id>/encoding_audit.json` and its
`.sha256`, read back through both. It also holds the section-by-section comparison of this run's inventory record
with the verified Commit 2 record (read in Cell 17, never written). On the machine that wrote that record only
`duplicates.callers` should differ (this step's modules import the feature code); `accepted_artifacts`,
`training_frames` and the block-cache scheme digest also differ where the artifacts are absent, the platform
differs or the checkout's line endings differ.

In [ ]:
# ============================================================
# CELL 21: WRITE THE ENCODING AUDIT RECORD
# ============================================================
INVENTORY_VS_COMMIT2 = pd.DataFrame(RR.compare_records(RECORD, COMMIT2))
AUDIT_PATH = RR.write_run_record(RUN, "encoding_audit", {
    "scope": "PR 42 Commit 3 Step 3A: encoding construction and leakage audit (training side)",
    "question": EA.QUESTION,
    "mechanisms": dict(EA.MECHANISMS),
    "time_index": {"source_order": dict(EA.SOURCE_ORDER), "resolution": dict(EA.RESOLUTION),
                   "source_checks": SOURCE_CHECKS.to_dict("records"),
                   "checks": TIME_CHECKS.to_dict("records"), "periods": PERIODS.to_dict("records"),
                   "contact_days": int(TIME_INDEX["day"].nunique()),
                   "recurring_day_keys": RECURRING_DAY_KEYS},
    "construction": {"encodings": dict(EA.CONSTRUCTION), "descendants": dict(EA.DESCENDANTS),
                     "stage1_settings": STAGE1_BUILD["settings"].to_dict("records"),
                     "stage1_bins": BINS.to_dict("records"), "stage1_cells": CELLS.to_dict("records")},
    "fitting_paths": {"paths": PATH_TABLE.to_dict("records"),
                      "plan_checks": PLAN_CHECKS.to_dict("records"),
                      "accepted_stage2_fold0": ROUTER_PLAN.to_dict("records")},
    "isolation": {"checks": ISOLATION.to_dict("records"),
                  "accepted_fits": ACCEPTED_FITS.to_dict("records"),
                  "mlp_early_stopping": {
                      "order": EARLY_STOPPING["order"],
                      "isolated_split": EARLY_STOPPING["isolated_split"],
                      "outer_isolation": EARLY_STOPPING["outer_isolation"],
                      "finding": EARLY_STOPPING["finding"],
                      "val_fraction": EARLY_STOPPING["val_fraction"],
                      "random_state": EARLY_STOPPING["random_state"],
                      "checks": EARLY_STOPPING["checks"].to_dict("records"),
                      "summary": EARLY_STOPPING["summary"].to_dict("records")}},
    "attribution": {"relations": list(EA.RELATIONS), "later": list(EA.LATER),
                    "summary": SUMMARY.to_dict("records"),
                    "coverage": COVERAGE.to_dict("records"),
                    "held_out_own_labels": HELD_OUT_OWN.to_dict("records"),
                    "fits": EA.compact_fits(AUDIT),
                    "final_mapping_by_period": BY_PERIOD.to_dict("records")},
    "provenance": PROVENANCE.to_dict("records"),
    "findings": {"by_path": FINDINGS.to_dict("records"),
                 "by_family": FAMILY_FINDINGS.to_dict("records"),
                 "mlp_early_stopping": EARLY_STOPPING["finding"]},
    "commit2_inventory": {"reference": "/".join(RR.COMMIT2_INVENTORY),
                          "reference_content_sha256": COMMIT2["content_sha256"],
                          "this_run_content_sha256": RECORD["content_sha256"],
                          "sections": INVENTORY_VS_COMMIT2.to_dict("records")},
})
AUDIT_RECORD = RR.read_run_record(RUN.run_id, "encoding_audit")
print(f"record  : {AUDIT_PATH.relative_to(BANK_ROOT)}")
print(f"content : {AUDIT_RECORD['content_sha256']}")
print(f"inventory vs Commit 2: {INVENTORY_VS_COMMIT2.loc[~INVENTORY_VS_COMMIT2['same'], ['section', 'part']].values.tolist() or 'identical'}")

## Step 3B — Past-only encodings and the temporal robustness protocol
Step 3A found the accepted encodings fold-isolated (mechanism b absent) but looking ahead in time (mechanism c).
Step 3B compares, for Stage 1, protocols without it. A **past-only** encoding of a row contacted on day d uses
only labels that could have been known then: the production `LRFeatureEngineer` fitted on the context's rows
contacted before day d, less an outcome lag, so the bin edges, bin rates, median fallback, cell rates and global
prior all come from those rows. Same-day labels are never used. The accepted and past-only encodings are compared
with the accepted LR and MLP settings, on identical rows, under the accepted random 10-fold plan and under
chronological windows. The 2 × 2 on the windows' rows contrasts the two encoding protocols under both splits.
Each contrast is a difference of whole protocols: the past-only protocol removes same-day and later labels, but
also each training row's own label, fits its edges, rates and fallbacks on fewer rows and leaves the cold-start
rows out of its fits, and no contrast here isolates one of these. The module is `feature_research.temporal_validity`;
the review document is `research_logs/pr42/temporal_validity.md`.

## Cell 22 — Registration
`TV.REGISTRATION` states what is known when each row is scored, the rules (lag 0 is the primary candidate; lags 5
and 20 contact days and the monthly rule are outcome-lag sensitivities), the cold start and the history floor, the
fallbacks for unseen values, the chronological windows, the models, the comparisons and the contract criteria. Its
digest goes into the record. Version 1.0.0 was fixed before any result; 1.1.0 (`revisions`) was made after review of
those results: it adds the lag-consistent chronological arm and its regression, names every contrast as a protocol
difference and changes the Step 3A record check, and changes no 1.0.0 comparison. The windows come from the time
index: 2008-05..2008-08 train only, then consecutive windows of at least 1,500 rows.

In [ ]:
# ============================================================
# CELL 22: REGISTRATION — rules, cold start, windows, comparisons
# ============================================================
REGISTRATION_DIGEST = TV.registration_digest()
WINDOWS = TV.window_table(TIME_INDEX, TRAIN.y_train)
print(f"{TV.REGISTRATION['id']} {TV.REGISTRATION['version']}  sha256 {REGISTRATION_DIGEST}")
print(f"cold start: the first {TV.COLD_START_DAYS} contact days (plus the lag); every state used holds "
      f">= {TV.MIN_STATE_ROWS} rows and >= {TV.MIN_STATE_POSITIVES} positives")
print(pd.DataFrame([{"rule": k, **v} for k, v in TV.RULES.items()]).to_string(index=False))
WINDOWS

## Cell 23 — Predictive runs
`TV.evaluate` fits, per fold of the accepted plan (`build_fold_designs`, 10 folds, seed 42) and per chronological
window:

- **LR**, every encoding: accepted (the fold's designs; the window fit), past-only under each rule (each fold's
  training rows are its own context), and in the windows `past_frozen` (predicted rows encoded by the cutoff state)
  and `past_updating` (predicted rows encoded at their own contact day, the model frozen).
- **Outcome lag in the windows**, two ways. `past_updating` under lag 5, lag 20 or monthly keeps the lag-0 model,
  trained on every pre-cutoff outcome, and slows only the prediction-time encodings: a lag-0 model whose incoming
  features update more slowly. `lag_consistent` operates under the delay throughout: the scaler and the model are
  fitted on the training rows whose outcomes are known at the cutoff under the lag (`TV.known_at_cutoff`), with the
  lag's encodings. At lag 0 the two are the same model (checked).
- **MLP**, accepted and primary past-only encodings, each with the accepted early stopping (the subset's labels are
  inside the encodings and the scaler, Step 3A) and with an **isolated** split: the estimator's own split is drawn
  first, the encoders and scaler are fitted without the subset, and every fit checks that the estimator stopped on
  those rows.

Rows a rule cannot encode (cold start) are left out of its fits and of its comparisons. Takes several minutes.

In [ ]:
# ============================================================
# CELL 23: PREDICTIVE RUNS — random 10-fold plan and chronological windows
# ============================================================
EVALUATION = TV.evaluate(TRAIN.X_train, TRAIN.y_train, TIME_INDEX)
print(f"{len(EVALUATION['scores'])} protocols scored; {len(EVALUATION['notes'])} MLP fits")

## Cell 24 — The past-only states against production, and availability
Every state a rule uses is the production encoder fitted on that state's rows: `TV.verification` fits
`LRFeatureEngineer` on exactly those rows and requires identical values on the rows that use the state, including
the lagged and monthly uses where the fallbacks occur. Availability per rule and scope: the rows a rule encodes,
the cold start, the euribor median fallback (a value outside the bins of the rows seen so far), unseen day × month
cells, the history behind each state, and how often the values differ from the accepted encoding's.

In [ ]:
# ============================================================
# CELL 24: PAST-ONLY STATES VERSUS PRODUCTION, AND AVAILABILITY
# ============================================================
VERIFICATION = TV.verification(TRAIN.X_train, TRAIN.y_train, TIME_INDEX, EVALUATION)
print(TV.require(VERIFICATION, "past-only states equal the production encoder"))
print(VERIFICATION[["context", "rule", "days", "states_compared", "rows_compared",
                    "fallback_values_compared", "max_abs_diff"]].to_string(index=False))
AVAILABILITY = TV.availability_tables(EVALUATION)
AVAILABILITY_CHECKS = TV.availability_checks(AVAILABILITY)
print(TV.require(AVAILABILITY_CHECKS, "state history floors"))
PRIMARY_ALL = EVALUATION["encodings"][f"all|{TV.PRIMARY}"]
AVAILABILITY_BY_PERIOD = TV.availability_by_period(PRIMARY_ALL, TIME_INDEX, PRIMARY_ALL["rows"])
AVAILABILITY[["scope", "rule", "rows", "available", "available_share", "euribor_fallback_rows",
              "cell_unseen_rows", "min_history_rows", "min_bins",
              "subscribe_rate_by_euribor_bin_rows_differ", "subscribe_rate_by_dow_month_rows_differ"]]

## Cell 25 — Changing an unavailable label cannot change an earlier encoding
For each rule, in the full-history context, a random fold's context and an isolated early-stopping context, at 40
contact days spread over the timeline: flipping every label unavailable to that day's encodings (the same day,
later days, inside the lag, the same period under the monthly rule) leaves them unchanged; flipping the same day's
labels alone leaves them unchanged; flipping the available labels changes them, so the check can fail. For
contrast, the accepted rule's encodings change when the same day's and later labels flip. Cell 23 adds the same
check on a deployed model's scores (first window), that the frozen predicted encodings equal the window fit's, and
the lag-consistent regression: in the first and the last window, flipping the pre-cutoff outcomes still unavailable
under lag 5 or lag 20 leaves the lag-consistent model (scaler, coefficients, intercept) and the deployed scores fixed
before those outcomes arrive unchanged, flipping known outcomes changes the model, and the same flip changes the
lag-0 model the prediction-time-only arm uses.

In [ ]:
# ============================================================
# CELL 25: INVARIANCE — flips of unavailable labels
# ============================================================
INVARIANCE = TV.invariance_checks(TRAIN.X_train, TRAIN.y_train, TIME_INDEX, TV.invariance_contexts(EVALUATION))
print(TV.require(INVARIANCE, "past-only encodings ignore unavailable labels"))
print(TV.require(EVALUATION["chrono_checks"], "chronological windows and deployed scores"))
INVARIANCE

## Cell 26 — Predictive comparisons: LR
First, the accepted protocol reproduces PR 39 (the mean fold ROC-AUC of the accepted LR and MLP on the accepted
designs). Then every registered comparison, paired DeLong on the rows both protocols scored (`b − a`, 95%
interval), pooled, per segment and as the mean over windows, and the 2 × 2 on the windows' rows: **A**
random/accepted, **B** random/past-only, **C** chronological/accepted, **D** chronological/past-only updating.
Protocol differences: past-only versus accepted, B − A and D − C; chronological versus random, C − A and D − B.
Each is a difference of whole protocols, consistent with but not isolating any one change, and an interval that
includes 0 leaves a difference unresolved rather than showing it absent. Pooled ROC-AUC mixes five models' score
levels across windows whose prevalence ranges from 7% to 49%; the unweighted mean over windows measures ranking
within them. Both are reported, and they can disagree in sign.

In [ ]:
# ============================================================
# CELL 26: PREDICTIVE COMPARISONS — LR
# ============================================================
REPRODUCTION = TV.reproduction_checks(EVALUATION)
print(TV.require(REPRODUCTION, "accepted protocols reproduce PR 39"))
PROTOCOLS = TV.protocol_table(EVALUATION)
COMPARED = TV.comparisons(EVALUATION)
FACTORIAL = {arm: TV.factorial(EVALUATION, arm) for arm in ("lr", "mlp")}
COLUMNS = ["question", "scope", "auc_a", "auc_b", "diff", "ci_low", "ci_high", "rows", "positives"]
SCOPES = ["pooled", "mean over windows"]
print(FACTORIAL["lr"]["cells"].drop(columns="protocol").round(4).to_string(index=False))
print(FACTORIAL["lr"]["effects"].round(4).to_string(index=False))
print(EVALUATION["lag_consistent_fits"].to_string(index=False))
COMPARED[(COMPARED["arm"] == "lr") & COMPARED["scope"].isin(SCOPES)][COLUMNS].round(4)

## Cell 27 — Predictive comparisons: MLP and its early stopping
The MLP's stopping epochs and best validation ROC-AUC under the accepted and the isolated early stopping, the
difference isolated − accepted (same encoding protocol and rows), and the 2 × 2 protocol differences with isolated
early stopping, so that none of them carries the Step 3A internal-validation issue.

In [ ]:
# ============================================================
# CELL 27: PREDICTIVE COMPARISONS — MLP, ACCEPTED VERSUS ISOLATED EARLY STOPPING
# ============================================================
EARLY_STOPPING_FITS = TV.early_stopping_table(EVALUATION)
print(EARLY_STOPPING_FITS.round(4).to_string(index=False))
print(FACTORIAL["mlp"]["cells"].drop(columns="protocol").round(4).to_string(index=False))
print(FACTORIAL["mlp"]["effects"].round(4).to_string(index=False))
COMPARED[(COMPARED["arm"] == "mlp") & COMPARED["scope"].isin(SCOPES)][COLUMNS].round(4)

## Cell 28 — Contract decisions and the Stage 2–3 families
The registered criteria applied to each candidate: realizable (Cell 25), values differing from the accepted
encoding's on at least 1% of rows, available for at least 95% of training rows. A past-only rule is a new feature
definition, so it needs its own versioned contract with distinct feature names; the accepted PR 39 contracts,
artifacts and records are unchanged. `TV.candidate_contract()` is the proposed contract (not registered). The
Stage 2–3 families are extended by construction and by the Step 3A attribution, not by experiment: the table says
which conclusions rest on direct evidence.

In [ ]:
# ============================================================
# CELL 28: CONTRACT DECISIONS AND THE STAGE 2-3 FAMILIES
# ============================================================
DECISIONS = TV.contract_decisions(AVAILABILITY[AVAILABILITY["scope"] == "random held-out rows"],
                                  INVARIANCE, COMPARED)
CANDIDATE_CONTRACT = TV.candidate_contract()
STAGE23 = TV.stage23_extension(SUMMARY)
print(DECISIONS.round(4).to_string(index=False))
print(f"proposed contract {CANDIDATE_CONTRACT['id']} {CANDIDATE_CONTRACT['version']}: "
      f"{CANDIDATE_CONTRACT['features']} (names {CANDIDATE_CONTRACT['fingerprint']}, definition "
      f"{CANDIDATE_CONTRACT['definition_fingerprint']}); accepted {CANDIDATE_CONTRACT['accepted_contract']}")
STAGE23

## Cell 29 — The PR 49 protocol and the interim finding
`TV.PR49_PROTOCOL` is the design PR 49 executes: the chronological split, what is fitted at each cutoff and what
may update after it, how prevalence shift is reported for calibration, thresholds, abstention and the Stage 2
obligations, and how a missed obligation is reported. `TV.stage2_prevalence_projection` shows what a pure prior
shift alone does to the frozen routing's obligations: the accepted routers' training OOF class-conditional
routing rates (PR 38 report, no artifact read) held fixed, the prevalence moved to each window's, the projected
counts observed with `observe_routing` against the frozen contract, exactly as PR 49 reports a window.
`TV.interim_finding` states what PR 43 may claim from the frozen PR 39 evidence, what PRs 44–45 need and what
stays open for PR 49, with this run's numbers.

In [ ]:
# ============================================================
# CELL 29: PR 49 PROTOCOL AND INTERIM FINDING
# ============================================================
PREVALENCES = dict(zip(WINDOWS["segment"], WINDOWS["prevalence"]))
PROJECTION = TV.stage2_prevalence_projection(PREVALENCES)
INTERIM = TV.interim_finding(COMPARED, FACTORIAL, DECISIONS, PROJECTION)
for key in ("split", "updated_after_cutoff", "prevalence_shift", "prior_shift_attribution",
            "missed_obligation_report"):
    print(f"{key}: {TV.PR49_PROTOCOL[key]}")
print(PROJECTION["bands"].round(4).to_string(index=False))
for key in ("finding", "pr43", "pr44_45", "pr49_open"):
    print(f"\n{key}: {INTERIM[key]}")
for key, text in INTERIM["interpretation"].items():
    print(f"\ninterpretation, {key}: {text}")
PROJECTION["projection"].round(4)

## Cell 30 — Research record: temporal validity
The run's third record, beside the first two: `research_logs/pr42/<run_id>/temporal_validity.json` and its
`.sha256`, read back through both. First, this run's Step 3A records are compared with the accepted Step 3A run
(`TV.STEP3A_RECORD`, `c3_20261122_201830`), and the applicable checks are enforced **before** the record is
written. If that run's directory is absent, the comparison is reported as unavailable. If it exists, it must hold
both records and their sidecars; a missing file fails the integrity check in any environment. Each record's
sidecar and content digest are checked, and that the digest is the accepted one. Exact reproduction is required
only where this run matches the accepted run's environment (operating system, platform, Python, interpreter and
package versions) and the same accepted artifacts were available to both inventories; on any other checkout, such
as Linux after the accepted run is committed, it is reported with the reason and not required. The accepted run is
read, never written.

In [ ]:
# ============================================================
# CELL 30: WRITE THE TEMPORAL VALIDITY RECORD
# ============================================================
STEP3A = TV.step3a_reproduction({"feature_inventory": RECORD, "encoding_audit": AUDIT_RECORD},
                                RUN.environment)
STEP3A_CHECKS = TV.step3a_checks(STEP3A)
print(STEP3A.drop(columns=["accepted_content_sha256"]).to_string(index=False))
print(TV.require(STEP3A_CHECKS, "accepted Step 3A record (integrity; reproduction where required)"))
TEMPORAL_PATH = RR.write_run_record(RUN, "temporal_validity", {
    "scope": "PR 42 Commit 3 Step 3B: past-only Stage 1 encodings and the temporal robustness protocol",
    "registration": {"content": TV.REGISTRATION, "sha256": REGISTRATION_DIGEST,
                     "rules": TV.RULES, "cold_start_days": TV.COLD_START_DAYS,
                     "min_state_rows": TV.MIN_STATE_ROWS, "min_state_positives": TV.MIN_STATE_POSITIVES,
                     "windows": WINDOWS.to_dict("records")},
    "settings": {k: v for k, v in EVALUATION["settings"].items()},
    "verification": VERIFICATION.to_dict("records"),
    "availability": {"tables": AVAILABILITY.to_dict("records"),
                     "checks": AVAILABILITY_CHECKS.to_dict("records"),
                     "primary_by_period": AVAILABILITY_BY_PERIOD.to_dict("records")},
    "invariance": {"encodings": INVARIANCE.to_dict("records"),
                   "chronological": EVALUATION["chrono_checks"].to_dict("records")},
    "predictive": {"reproduction": REPRODUCTION.to_dict("records"),
                   "protocols": PROTOCOLS.to_dict("records"),
                   "comparisons": COMPARED.to_dict("records"),
                   "factorial": {arm: {"cells": f["cells"].to_dict("records"),
                                       "effects": f["effects"].to_dict("records"),
                                       "rows": f.get("rows"), "positives": f.get("positives"),
                                       "windows": f.get("windows")} for arm, f in FACTORIAL.items()},
                   "fold_roc_auc": EVALUATION["fold_roc_auc"],
                   "lag_consistent_fits": EVALUATION["lag_consistent_fits"].to_dict("records"),
                   "mlp_fits": EVALUATION["notes"].to_dict("records"),
                   "mlp_early_stopping": EARLY_STOPPING_FITS.to_dict("records")},
    "contracts": {"decisions": DECISIONS.to_dict("records"), "candidate": CANDIDATE_CONTRACT},
    "stage23_extension": STAGE23.to_dict("records"),
    "pr49_protocol": TV.PR49_PROTOCOL,
    "stage2_prior_shift": {"contract": PROJECTION["contract"], "routing": PROJECTION["routing"],
                           "nominal_rows": PROJECTION["nominal_rows"],
                           "bands": PROJECTION["bands"].to_dict("records"),
                           "projection": PROJECTION["projection"].to_dict("records")},
    "interim_finding": INTERIM,
    "step3a_reproduction": {"accepted": TV.STEP3A_RECORD, "environment_keys": list(TV.ENVIRONMENT_KEYS),
                            "comparison": STEP3A.to_dict("records"),
                            "checks": STEP3A_CHECKS.to_dict("records")},
})
TEMPORAL_RECORD = RR.read_run_record(RUN.run_id, "temporal_validity")
print(f"record  : {TEMPORAL_PATH.relative_to(BANK_ROOT)}")
print(f"content : {TEMPORAL_RECORD['content_sha256']}")

## Step 4A — Stage 2 NSD/ECI binning drift: reproduction and diagnosis
PR 39 Step 3A retained both bin families but left the lift drift open: the figures quoted when the thresholds were
locked (`eci_warm` 1.49–1.67×, `eci_hot` 3.53×, `nsd_elevated` 1.04–1.43×) against the fold-isolated out-of-fold
lifts (about 1.03×, 2.50× and 2.49×, the last with one fold far below the others). Step 4A traces each figure to
what it measures, reproduces the PR 39 evidence by import, and separates threshold placement, encoder behaviour,
population (contact-period mix) and in-sample label dependence wherever the evidence permits. Commit 3's look-ahead
shares enter as label provenance, not as an effect. Diagnosis only: no threshold is proposed, no binning procedure is
defined and nothing is selected (Step 4B). The frozen binning, `stage2_binning` and its accepted record are read,
never changed. The module is `feature_research.binning_drift`; the review document is
`research_logs/pr42/binning_drift.md`.

## Cell 31 — Where the figures come from
The PR 39 Step 3A record, read after its pinned SHA-256. The historical figures are Cells 13C–13D of the exploratory
notebook at commit `1ce1649` (2026-02-28), the run the thresholds were locked from: decile lifts, and the locked
bins' lifts printed beside them, on all 41,188 rows (before the cascades' split existed) with an encoder fitted on
those same rows. They are transcribed in `BD.HISTORICAL`, checked against `git show 1ce1649:…` where the commit is in
the checkout (`unavailable` in a shallow clone; a mismatch stops the run), and never recomputed. `implied_cut_rows`
uses the printed counts to place each frozen cut against the rounded decile edge it was read from. The exploratory
notebook's Cells 15A–15B ran the same analysis on the training rows. Their accepted printed outputs are read from the
notebook's pinned Git blob (`BD.EXPLORATORY`, committed in PR 36), never from the working copy, which Commit 6 may
clear. Only if Git lacks the blob is the working copy used, and only when its SHA-256 is the pinned one; otherwise the
status is `unavailable` and Cell 36 reports those comparisons as not checked. A mismatch stops the run.

In [ ]:
# ============================================================
# CELL 31: FIGURES AND THEIR PROVENANCE
# ============================================================
PR39_BINNING, PR39_BINNING_SOURCE = BD.read_pr39_record()
HISTORICAL_CHECK = BD.verify_historical()
if HISTORICAL_CHECK["status"] == "mismatch":
    raise AssertionError(f"historical transcription: {HISTORICAL_CHECK['detail']}")
IMPLIED_CUTS = BD.implied_cut_rows()
EXPLORATORY_OUT = BD.exploratory_outputs()
if EXPLORATORY_OUT["status"] == "mismatch":
    raise AssertionError(f"exploratory outputs: {EXPLORATORY_OUT['detail']}")
FIGURES = BD.figure_provenance(PR39_BINNING)
print(f"PR 39 record {PR39_BINNING_SOURCE['run_id']} (SHA-256 pinned): outcome {PR39_BINNING_SOURCE['outcome']}")
print(f"historical transcription: {HISTORICAL_CHECK['status']} ({HISTORICAL_CHECK['detail']})")
print(BD.require(pd.DataFrame(HISTORICAL_CHECK["arithmetic"]), "historical arithmetic"))
print(f"exploratory printed outputs: {EXPLORATORY_OUT['status']} ({EXPLORATORY_OUT['detail']})")
print(IMPLIED_CUTS.to_string(index=False))
FIGURES[["bin", "figure", "estimand", "intervals", "rows", "encoder_fit", "reproduced_here"]]

## Cell 32 — PR 39 Step 3A reproduced by import, and the source values
`BD.stage2_side` builds the shared outer plan on the training rows (`make_folds(y, 5, 42)`, the PR 39 fold
fingerprint). `BD.reproduce_pr39` runs `stage2_binning.runner.eb_evidence` (a fresh engineer per outer fold, binned
through the production path, beside the in-sample table), `evidence.triggers` (T1–T4) and `evidence.decide`, and
requires each to equal the accepted record. `BD.source_values` keeps the row-level values: out of fold from
`stage2_binning.runner.oof_sources`, in sample from Cell 7's `TRAINING_FRAMES["stage2_sources"]` (the production
engineer fitted on all training rows); their tables must be E-B's. The module fits no engineer itself and imports no
feature-engineering module, so the Commit 2 caller scan, and the Step 3A digests Cell 30 checks, are unchanged.

In [ ]:
# ============================================================
# CELL 32: PR 39 STEP 3A REPRODUCED, AND THE SOURCE VALUES
# ============================================================
STAGE2_SIDE = BD.stage2_side(TRAIN.X_train, TRAIN.y_train)
PR39_REPRO = BD.reproduce_pr39(STAGE2_SIDE, PR39_BINNING)
print(BD.require(PR39_REPRO["checks"], "PR 39 Step 3A reproduced"))
SOURCES_4A = BD.source_values(STAGE2_SIDE, TRAINING_FRAMES["stage2_sources"], PR39_REPRO["eb"])
print(BD.require(SOURCES_4A.checks, "source values"))
pd.concat([PR39_REPRO["checks"], SOURCES_4A.checks], ignore_index=True)

## Cell 33 — Held-out labels never reach their own encodings
For each outer fold, every held-out label is flipped and the out-of-fold sources are recomputed through
`oof_sources`. The fold's own NSD and ECI must not move. As a control, other folds' values must move (the flipped
labels are in their fits), so the check can fail. Five recomputations of the five fold fits.

In [ ]:
# ============================================================
# CELL 33: HELD-OUT LABEL FLIPS
# ============================================================
FLIP_CHECKS = BD.heldout_label_checks(STAGE2_SIDE)
print(BD.require(FLIP_CHECKS, "held-out labels never reach their own encodings"))
FLIP_CHECKS

## Cell 34 — Macro states and the encoder comparison
NSD reads only (euribor3m, nr.employed, emp.var.rate), through its scaler and kernel; ECI reads the same three
columns through their quantile bins. Under any one fitted engineer both are functions of this **macro state**, so
rows sharing a state cross a cut together. `BD.macro_states` numbers the states in source order, with their periods.
`BD.fold_fit_values` evaluates each outer fold's engineer on its own fit rows through `oof_sources` itself (a
two-group side whose group-1 engineer is fitted on exactly that fold's fit rows; its label fingerprint must be
E-B's). `BD.state_values` checks that each encoder's value is constant within a state and that both sources of a
fold engineer's values agree.

`BD.encoder_comparison` then compares three lifts per bin with the frozen cut points: in sample (the all-rows engineer
on its rows), fold fit (each fold's engineer on its own fit rows; every row counts four times) and out of fold (each
fold's engineer on its held-out rows). Two **descriptive** contrasts add up to `oof_minus_in_sample`:
`fold_fit_minus_in_sample` (different encoders, each on the rows that built it) and `heldout_minus_fold_fit` (the
same encoders on different rows). Neither isolates a cause. `heldout_minus_fold_fit` is not the effect of a row's own
label: held-out and fit rows differ in composition, and per-fold differences of opposite sign cancel in the pool. The
direct evidence of what an encoder does is the macro states it assigns to each bin (Cells 37–38) and the migration
table, which crosses each row's in-sample bin with its out-of-fold bin.

In [ ]:
# ============================================================
# CELL 34: MACRO STATES AND THE ENCODER COMPARISON
# ============================================================
STATES = BD.macro_states(TRAIN.X_train, TIME_INDEX, TRAIN.y_train)
FOLD_FIT = BD.fold_fit_values(STAGE2_SIDE, STATES, PR39_REPRO["eb"])
STATE_VALUES = BD.state_values(STATES, SOURCES_4A, STAGE2_SIDE, FOLD_FIT)
print(BD.require(STATE_VALUES["checks"], f"{len(STATES.table)} macro states"))
ENCODER = BD.encoder_comparison(STAGE2_SIDE, SOURCES_4A, STATES, STATE_VALUES)
MOVED = ENCODER["migration"][ENCODER["migration"]["in_sample_bin"] != ENCODER["migration"]["oof_bin"]]
print(MOVED[["family", "in_sample_bin", "oof_bin", "rows", "lift"]].round(3).to_string(index=False))
ENCODER["decomposition"].round(3)

## Cell 35 — Support, uncertainty and fold stability
The frozen bins in sample and out of fold, pooled and per fold, from `stage2_binning.evidence.bin_table`: support,
subscribers, rate, lift and its Wilson 95% interval. **Estimand** (`BD.WILSON_ESTIMAND`): lift is the bin's rate over
the base rate of the same rows, and the interval is the Wilson interval of the rate divided by that base rate. It
covers sampling of the evaluated rows with the encoders, cut points and base rate fixed, not fold-to-fold encoder
variation, which the per-fold rows show. T1–T4 are PR 39's, reproduced in Cell 32.

In [ ]:
# ============================================================
# CELL 35: SUPPORT, UNCERTAINTY AND FOLD STABILITY
# ============================================================
SUPPORT = BD.support_table(PR39_REPRO["eb"])
STABILITY = BD.fold_stability(PR39_REPRO["eb"], PR39_REPRO["triggers"])
print(STABILITY.round(3).to_string(index=False))
SUPPORT[SUPPORT["bin"].isin(list(BD.FOCAL))].round(3)

## Cell 36 — Decile lift and the historical thresholds
Deciles as the exploratory lift analysis forms them (`BD.DECILE_DEFINITION`: `qcut(10, duplicates="drop")`, ties
kept together, so row counts are unequal): in sample with in-sample edges, and out of fold with the pooled
out-of-fold edges applied to each fold. Each decile lists the frozen bins its rows fall in and any frozen cut inside
it. The in-sample deciles and bins must equal the exploratory notebook's accepted printed Cells 15A–15B (read in
Cell 31 from the pinned blob; reported as not checked if that source is unavailable), and both kinds must equal
`compute_lift`'s. Then the **historical candidate thresholds**, the nine printed Cell 13C decile edges per
family that the frozen cuts were read from, applied unchanged to the training-side encodings beside the historical
counts and lifts. They are described, not evaluated as candidates (Step 4B).

In [ ]:
# ============================================================
# CELL 36: DECILE LIFT AND THE HISTORICAL THRESHOLDS
# ============================================================
DECILES = BD.decile_tables(STAGE2_SIDE, SOURCES_4A)
DECILE_CHECKS = pd.concat([BD.exploratory_reproduction(DECILES, PR39_REPRO["eb"], EXPLORATORY_OUT),
                           BD.compute_lift_parity(STAGE2_SIDE, SOURCES_4A, DECILES)], ignore_index=True)
print(BD.require(DECILE_CHECKS, "deciles"))
print(DECILES["table"][["family", "kind", "decile", "display", "rows", "lift", "frozen_bins",
                        "frozen_cuts_inside"]].round(3).to_string(index=False))
HISTORICAL_CUTS = BD.historical_thresholds(STAGE2_SIDE, SOURCES_4A)
HISTORICAL_CUTS["table"].round(3)

## Cell 37 — Contact periods
Bin membership, prevalence and support by contact period (Cell 14's time index), in sample and out of fold with each
fold's rows. The within-period ratio is the bin's rate over its period's rate. Per bin, the period-standardized ratio
divides observed subscribers by expected ones (the bin's rows at their periods' rates). Near 1, the period mix
reproduces the bin's aggregate subscriber count; within-period differences of opposite sign can cancel in it, so the
absolute within-period deviation (as a share of expected), the within-period ratio range (periods with at least
`BD.PERIOD_MIN_ROWS` rows) and the periods the bin shares with other bins are shown beside it. The **straddling
states** are macro states whose frozen bin differs between the six training-side encoders, with the cut their value
range crosses. Commit 3's attribution (`EA.audit_paths` on the Stage 2 final fit and outer folds) gives each row's
share of label mass (influence, for ECI) from later contact days, here by bin. These shares are label provenance:
similar means can come from different contributing labels and give different encoded values, so their effect on a
lift gap is left unresolved.

In [ ]:
# ============================================================
# CELL 37: CONTACT PERIODS, STRADDLING STATES AND LABEL PROVENANCE
# ============================================================
PERIODS_4A = BD.period_evidence(STAGE2_SIDE, SOURCES_4A, TIME_INDEX)
STRADDLING = BD.straddling_states(STATES, STATE_VALUES)
PROVENANCE_4A = BD.label_provenance_by_bin(TRAIN.X_train, TRAIN.y_train, TIME_INDEX, STAGE2_SIDE, SOURCES_4A)
print(BD.require(PROVENANCE_4A["checks"], "Commit 3 attribution on this plan"))
FOCAL_PERIODS = PERIODS_4A["composition"][PERIODS_4A["composition"]["bin"].isin(list(BD.FOCAL))]
print(FOCAL_PERIODS.round(3).to_string(index=False))
print(STRADDLING["by_bin"].to_string(index=False))
print(STRADDLING["states"].head(15).round(4).to_string(index=False))
PROVENANCE_4A["table"].round(3)

## Cell 38 — Findings
`BD.focal_assignments` establishes the row- and state-level facts the findings state, on row masks and on the
per-state, per-encoder assignments of Cell 34: which states each encoder assigns to the focal bins, whether two row
sets are identical, and the folds of every row that changes bin. Then one finding per focal bin: the best-supported
explanation, the causes the comparisons leave entangled, and each claim with the numbers it rests on. Each claim says
only what its check establishes. Every claim must hold in this run. The review document states these claims, so a run
in which one fails stops here.

In [ ]:
# ============================================================
# CELL 38: FINDINGS
# ============================================================
ASSIGNMENTS = BD.focal_assignments(STAGE2_SIDE, SOURCES_4A, TIME_INDEX, STATES, STATE_VALUES, DECILES)
FINDINGS_4A = BD.findings(eb=PR39_REPRO["eb"], historical_check=HISTORICAL_CHECK, implied=IMPLIED_CUTS,
                          encoder=ENCODER, deciles=DECILES, historical_cuts=HISTORICAL_CUTS,
                          periods=PERIODS_4A, straddling=STRADDLING, provenance=PROVENANCE_4A,
                          assignments=ASSIGNMENTS)
CLAIMS_4A = BD.claims_table(FINDINGS_4A)
print(BD.require(CLAIMS_4A, "Step 4A claims"))
for f in FINDINGS_4A:
    print(f"\n{f['bin']}: {f['observation']}\n  {f['explanation']}\n  entangled: {f['entangled']}")
CLAIMS_4A

## Cell 39 — Research record: binning drift
The run's fourth record, beside the first three: `research_logs/pr42/<run_id>/binning_drift.json` and its `.sha256`,
read back through both.

In [ ]:
# ============================================================
# CELL 39: WRITE THE BINNING DRIFT RECORD
# ============================================================
DRIFT_PATH = RR.write_run_record(RUN, "binning_drift", BD.record_sections(
    pr39_provenance=PR39_BINNING_SOURCE, historical_check=HISTORICAL_CHECK, implied=IMPLIED_CUTS,
    exploratory=EXPLORATORY_OUT, figures=FIGURES, reproduction=PR39_REPRO, sources=SOURCES_4A,
    flips=FLIP_CHECKS, states=STATES, state_vals=STATE_VALUES, encoder=ENCODER, support=SUPPORT,
    stability=STABILITY, deciles=DECILES, decile_checks=DECILE_CHECKS, historical_cuts=HISTORICAL_CUTS,
    periods=PERIODS_4A, straddling=STRADDLING, provenance=PROVENANCE_4A, assignments=ASSIGNMENTS,
    found=FINDINGS_4A))
DRIFT_RECORD = RR.read_run_record(RUN.run_id, "binning_drift")
print(f"record  : {DRIFT_PATH.relative_to(BANK_ROOT)}")
print(f"content : {DRIFT_RECORD['content_sha256']}")

## Step 4B — Fold-fitted NSD/ECI binning and `eci_warm` as a filter
Step 4A left three questions: whether `eci_warm` helps a partner rule identify subscribers or exclude false positives
at usable support; whether a fold-fitted procedure can find a useful `eci_hot` upper tail with stable support; and
whether a procedure can make `nsd_elevated` less sensitive to whole macro states crossing its cuts. Step 4B
**registers** the comparison first (Cell 40), then fits each procedure inside every outer fold and compares it with
the frozen PR 39 bins on identical held-out rows. Each held-out row is scored with its fold's engineer and its fold's
cuts; label-using choices use inner cross-fitted values on the fit side only. Every procedure also reads fixed
reference inputs formed on all training rows (Cell 41), so none is fully nested. Cell 47 is a separately labelled,
post-registration sensitivity check for the advancing procedure. The frozen binning, `stage2_binning`
and the accepted PR 39 and Step 4A records are read, never changed: a candidate that earns it goes to PR 44, which
owns the router comparison. The module is `feature_research.binning_candidates`; the review document is
`research_logs/pr42/binning_candidates.md`.

## Cell 40 — The registration
`BC.REGISTRATION` (version 1.0.0) was fixed, and its SHA-256 pinned in `BC.REGISTERED_SHA256`, before any candidate
cut, partner rule or held-out statistic of one was computed. It separates what Step 4A had established (and which
hypothesis each finding motivated) from what this step fixes: the frozen comparator, three procedures
(`share_matched` and `gap_placed` for both families, label-free; `tail_selected` for ECI, label-using on the fit side),
the partner-rule selection, the rows and folds, the measures, the criteria A0–A4 for advancing a candidate, the
filter criteria, and how ties, sparse or empty bins, infeasible fits and lift against support are handled.
`BC.registration_checks` requires the pinned digest, the PR 39 record's in-sample supports, the frozen GLASS profile's
and shared contract's values, the unchanged frozen binning, and a partner pool GLASS can express. This cell runs
before any candidate is computed.

In [ ]:
# ============================================================
# CELL 40: STEP 4B REGISTRATION
# ============================================================
REGISTRATION_4B = BC.registration_digest()
REGISTRATION_CHECKS_4B = BC.registration_checks(PR39_BINNING)
print(BC.require(REGISTRATION_CHECKS_4B, "Step 4B registration"))
if STAGE2_SIDE.fold_fingerprint != BC.FOLD_FINGERPRINT:
    raise AssertionError(f"outer plan {STAGE2_SIDE.fold_fingerprint}, registered {BC.FOLD_FINGERPRINT}")
print(f"{BC.REGISTRATION['id']} {BC.REGISTRATION['version']}  sha256 {REGISTRATION_4B}")
print(f"version 1.0.0: {BC.REGISTRATION['revisions'][0]['status']}")
BC.registration_table()

## Cell 41 — Fold-fitted procedures: the learned cuts
`BC.heldout_sources` bins all eight out-of-fold sources of every training row (the partner rules' held-out bins),
through `oof_sources` again, and requires the fits and the NSD/ECI values to be Step 4A's. `BC.fit_folds` fits, for
each outer fold and from its fit side only: the inner cross-fitted sources (`oof_sources` on the fit side, plan
`make_folds(y_fit, 5, 42)`); each procedure's three cuts per family, placed between blocks of macro states on fold k's
engineer values (ties within 1e-12 form one block, so no cut splits a state); and the partner rules.
`BC.deployment_fits` fits each procedure on all training rows, for the refit-stability comparison only. The table
lists every fit's cuts, their fit-row shares, how each was chosen, and the fit-side rows and states per bin.

**Reference inputs (`BC.reference_inputs`).** The fits are fold-fitted given three fixed inputs, printed first:
`frozen_shares()` (the targets of `share_matched`, `gap_placed` and `tail_selected`'s first two cuts),
`window_halfwidths()` (`gap_placed`'s windows) and `_tail_range()` (`tail_selected`'s candidate tails). All three are
functions of `BC.FROZEN_SUPPORTS`, the frozen bins' in-sample supports on all 32,950 training rows under the all-rows
engineer, fitted with every training label. Each fold's held-out rows and labels therefore helped form them, and the
frozen cuts behind those supports were read from deciles on all 41,188 rows. The limitation is the frozen
comparator's, and it applies to every registered procedure, not only `share_matched`.

In [ ]:
# ============================================================
# CELL 41: FOLD-FITTED PROCEDURES AND THEIR CUTS
# ============================================================
print(BC.reference_inputs()[["input", "family", "values", "used_by"]].to_string(index=False))
HELDOUT_BINS = BC.heldout_sources(STAGE2_SIDE, SOURCES_4A)
print(BC.require(HELDOUT_BINS["checks"], "held-out sources"))
FOLD_FITS_4B = BC.fit_folds(STAGE2_SIDE, STATES, STATE_VALUES)
print(BC.require(pd.concat([ff["checks"] for ff in FOLD_FITS_4B], ignore_index=True), "inner cross-fits"))
ALL_ROWS_FITS = BC.deployment_fits(STAGE2_SIDE, STATES, STATE_VALUES, SOURCES_4A)
CUTS_4B = BC.cut_table(FOLD_FITS_4B, ALL_ROWS_FITS)
print(f"infeasible fits: {int((~CUTS_4B['feasible']).sum())}")
CUTS_4B[["family", "procedure", "fit", "cut_1", "cut_2", "cut_3", "share_1", "share_2", "share_3", "how_3",
         "bin_rows", "bin_states", "selected_tail_share"]].round(4)

## Cell 42 — Held-out labels never reach their fold's fits
For each outer fold, every held-out label is flipped and the fold is fitted again: inner values, cuts, the tail choice,
the partner rules and the Cell 47 sensitivity fit must not change. As a control, flipping every tenth fit-side label
must move the inner values, so the check can fail. Fold k's engineer, which supplies the scoring values, is Step 4A's;
Cell 33 flips its held-out labels. **The checks are conditional on the fixed reference inputs** (`BC.ISOLATION_SCOPE`):
the frozen cuts, `frozen_shares()`, `window_halfwidths()` and `_tail_range()` do not move when fold k's labels are
flipped. The flips therefore show that no fit-side computation reads fold k's held-out labels. They cannot show that
the constants are independent of fold k's rows and labels, and those constants are not.

In [ ]:
# ============================================================
# CELL 42: HELD-OUT LABEL FLIPS FOR THE FOLD FITS
# ============================================================
ISOLATION_4B = BC.isolation_checks(STAGE2_SIDE, STATES, STATE_VALUES, FOLD_FITS_4B)
print(BC.require(ISOLATION_4B, "held-out labels never reach their fold's fits"))
ISOLATION_4B

## Cell 43 — The comparison on identical held-out rows
Every procedure bins the same held-out rows with the same out-of-fold values; only the cuts differ. The comparator must
be E-B's out-of-fold binning row by row and table by table, and the frozen cuts through the candidates' apply path
must give the production bins. Per bin, pooled and per fold, `stage2_binning.evidence.bin_table` gives support,
subscribers, rate, lift and its Wilson 95% interval, and `evidence.triggers` PR 39's T1–T4. For the focal bins
(`nsd_elevated`, `eci_hot`): the fold range, the smallest fold support against the frozen bin's, the refit migration
(held-out rows whose focal membership differs between their fold's fit and the all-rows fit; for the frozen bins this
is Step 4A's migration), the triggers fired, and the overlap with the frozen focal bin.

In [ ]:
# ============================================================
# CELL 43: CANDIDATES AGAINST THE FROZEN BINS
# ============================================================
COMPARISON_4B = BC.compare(STAGE2_SIDE, SOURCES_4A, FOLD_FITS_4B, ALL_ROWS_FITS, PR39_REPRO["eb"])
print(BC.require(COMPARISON_4B["checks"], "the comparator is E-B"))
print(COMPARISON_4B["bins"][["family", "procedure", "bin", "support", "positives", "lift"]]
      .round(3).to_string(index=False))
COMPARISON_4B["focal"][["family", "bin", "procedure", "support", "lift", "lift_lower", "lift_upper",
                        "min_fold_lift", "max_fold_lift", "min_support_ratio", "migration_rows",
                        "triggers_fired", "overlap_with_frozen"]].round(3)

## Cell 44 — Does a gain depend on one fold or one contact period?
The focal bin's pooled lift for every procedure with each outer fold removed, and with each contact period of at
least `BC.PERIOD_MIN_ROWS` rows removed (Cell 14's periods), on the same remaining rows; and the period holding the
largest share of the focal bin's subscribers. Criterion A4 reads these rows. The removals overlap; they are not
independent replicates.

In [ ]:
# ============================================================
# CELL 44: FOLD AND PERIOD DEPENDENCE
# ============================================================
DEPENDENCE_4B = BC.dependence(STAGE2_SIDE, TIME_INDEX, COMPARISON_4B)
print(DEPENDENCE_4B["subscriber_sources"].round(3).to_string(index=False))
DEPENDENCE_4B["removals"].pivot_table(index=["family", "removed"], columns="procedure", values="lift").round(3)

## Cell 45 — `eci_warm` as a filter on partner rules
Each fold's partners are chosen on its fit side (inner cross-fitted bins): one- or two-literal conjunctions of non-ECI
bins that pass GLASS's per-rule precision and recall gates and the scaled support floors, ranked by the Wilson lower
bound of precision, at most five with fit-row Jaccard overlap at most 0.5. On the fold's held-out rows each partner is
evaluated alone, with `eci_warm` required and with it excluded (the frozen bin on the out-of-fold ECI). The two
variants must partition the partner's rows and add up to its counts; what each removes is counted. **Population:**
every held-out training row, a diagnostic population broader than Pass 2's (the remainder after a GLASS Pass 1 fit,
which no tracked artifact holds). The union obligations (precision 0.30, support 75) are applied as a screen, pooled
over the five folds; passing it is not compliance, and a promising conjunction still needs PR 44's router comparison.
**Support** (`BC.FILTER_SCOPE`): the rows and subscribers a variant keeps or removes are a standalone conjunction's. In
a router, the rows a filter drops can still be routed by other rules of the selected union, and Pass 1 changes the
population, so the router-level coverage effect can differ in size or direction. PR 44 measures it.

In [ ]:
# ============================================================
# CELL 45: ECI_WARM AS A FILTER
# ============================================================
FILTER_4B = BC.warm_filter(STAGE2_SIDE, HELDOUT_BINS, FOLD_FITS_4B)
print(BC.require(FILTER_4B["checks"], "the variants partition each partner"))
print(FILTER_4B["per_fold"][["fold", "slot", "partner", "alone_rows", "alone_precision", "required_rows",
                             "required_precision", "excluded_rows", "excluded_precision"]]
      .round(3).to_string(index=False))
FILTER_4B["by_slot"][["slot", "partners", "alone_rows", "alone_subscribers", "alone_precision",
                      "required_rows", "required_precision", "required_ratio", "required_folds_above_partner",
                      "required_removes_subscribers", "required_removes_non_subscribers", "required_effect",
                      "excluded_ratio", "excluded_effect"]].round(3)

## Cell 46 — The decision
`BC.decide` applies the registered criteria: per candidate A0 (feasible in every fold), A1 (no PR 39 trigger), A2
(focal support at least half the frozen bin's in every fold), A3 (the family's question) and A4 (the gain survives
removing any one fold or contact period); the first-ranked candidate meeting all five advances to PR 44, the others
are closed out with the reason. The frozen bins are kept in PR 42 in every case. `eci_warm` gets a separate verdict
as a filter. `BC.stated_checks` requires every verdict and figure the review document states to hold in this run.
These are the registered results; Cell 47's sensitivity does not change them.

In [ ]:
# ============================================================
# CELL 46: DECISION
# ============================================================
DECISION_4B = BC.decide(COMPARISON_4B, DEPENDENCE_4B, FILTER_4B)
STATED_4B = BC.stated_checks(DECISION_4B, COMPARISON_4B, FILTER_4B, DEPENDENCE_4B)
print(BC.require(STATED_4B, "the review document's statements"))
print(DECISION_4B["verdicts"].to_string(index=False))
print(f"eci_warm as a filter: {DECISION_4B['eci_warm_filter']['verdict']} "
      f"({DECISION_4B['eci_warm_filter']['direction']}; {DECISION_4B['eci_warm_filter']['slots_required_effect']} "
      f"of {DECISION_4B['eci_warm_filter']['slots']} slots)")
DECISION_4B["criteria"]

## Cell 47 — Post-registration sensitivity: fold-local reference inputs for NSD `gap_placed`
**Not registered; outside the ranking and the verdicts.** Added at review, after the registered results were seen
(`BC.SENSITIVITY`). Within each outer fold the frozen comparator cuts (0.0524 / 0.192 / 0.23, with their historical
provenance) are applied to that fold's own fit-side NSD values, its engineer's values of its fit rows. The fit-row
shares below the cuts are the fold-local targets, and the fit-row shares of the frozen bins give the window
half-widths, from those fit rows alone. `gap_placed`'s placement, the outer folds, the comparator and the held-out
evaluation rows are unchanged. The all-rows reference fit applies the same rule to all training rows. Reported beside
the registered result: the fold-local targets, the learned cuts, focal support, pooled and worst-fold lift, refit
migration, the rows whose membership differs from the registered fit, and whether the fold-0 repair and the zero
migration persist. The registered criteria are applied descriptively; they decide nothing here.

In [ ]:
# ============================================================
# CELL 47: POST-REGISTRATION SENSITIVITY (NOT REGISTERED)
# ============================================================
SENSITIVITY_ALL_ROWS = BC.sensitivity_all_rows(STAGE2_SIDE, STATES, STATE_VALUES)
SENSITIVITY_4B = BC.sensitivity(STAGE2_SIDE, SOURCES_4A, TIME_INDEX, FOLD_FITS_4B, SENSITIVITY_ALL_ROWS,
                                COMPARISON_4B, ALL_ROWS_FITS)
SENSITIVITY_CHECKS_4B = BC.sensitivity_checks(SENSITIVITY_4B)
print(BC.require(SENSITIVITY_CHECKS_4B, "the review document's sensitivity statements"))
print(f"{BC.SENSITIVITY['id']}: {BC.SENSITIVITY['status']}")
print(pd.DataFrame(SENSITIVITY_4B["per_fold"])[["fold", "fold_local_targets", "fold_local_halfwidths", "cuts",
                                                "registered_cuts"]].round(4).to_string(index=False))
print({k: v for k, v in SENSITIVITY_4B["persists"].items() if not k.endswith("definition")})
SENSITIVITY_4B["summary"][["procedure", "status", "support", "lift", "lift_lower", "lift_upper", "min_fold_lift",
                           "fold0_lift", "migration_rows", "differing_by_fold"]].round(3)

## Cell 48 — Research record: binning candidates
The run's fifth record, beside the first four: `research_logs/pr42/<run_id>/binning_candidates.json` and its
`.sha256`, read back through both. It holds the registration and its digest, every fit, the comparison, the
dependence rows, the filter evaluation, the decision and the limits.

In [ ]:
# ============================================================
# CELL 48: WRITE THE BINNING CANDIDATES RECORD
# ============================================================
CANDIDATES_PATH = RR.write_run_record(RUN, "binning_candidates", BC.record_sections(
    reg_checks=REGISTRATION_CHECKS_4B, heldout=HELDOUT_BINS, fold_fits=FOLD_FITS_4B, deployed=ALL_ROWS_FITS,
    cuts=CUTS_4B, isolation=ISOLATION_4B, comparison=COMPARISON_4B, dep=DEPENDENCE_4B, filt=FILTER_4B,
    decision=DECISION_4B, stated=STATED_4B, sens=SENSITIVITY_4B, sens_checks=SENSITIVITY_CHECKS_4B))
CANDIDATES_RECORD = RR.read_run_record(RUN.run_id, "binning_candidates")
print(f"record  : {CANDIDATES_PATH.relative_to(BANK_ROOT)}")
print(f"content : {CANDIDATES_RECORD['content_sha256']}")

## Step 5A — Feature analysis and Stage 3 simplification
Can the frozen PR 39 Stage 3 EBM be simplified so its contributions are easier to explain and more stable across
folds and contact periods, and what does each simplification cost in out-of-fold performance, calibration and
complementarity with Stage 1? The comparison is registered first (Cell 49). The baseline is the accepted 13-feature,
three-pair EBM, refitted on the training side through the existing Stage 3 code: `Stage3Block` with the production
feature pipeline refitted inside every fold, `EBMFactory` with the accepted hyperparameters, `OOFGenerator`,
`Stage3Calibrator` and `F2ThresholdSelector`. Four candidates follow the same protocol on the same rows. Accepted
artifacts are read only through the PR 38 boundary extractors, and only where they exist locally. The module is
`feature_research.stage3_simplification`; the review document is `research_logs/pr42/stage3_simplification.md`.

## Cell 49 — Registration and the baseline's sources
`SS.REGISTRATION` (version 1.0.0) was fixed, and its SHA-256 pinned, before any candidate was fitted; only the
baseline had been refitted once, to check parity. It records the baseline's sources, the parity checks, the rule that
identifies the remaining correlated pair, the four candidates and how their pairs are resolved, the past-only density
(definition, outcome-availability assumption, cold start, fold isolation and contact ordering), the measures, the
shape conventions, the complementarity counts and the nomination rules. `SS.registration_checks` requires the digest
and checks the baseline against its sources: the 13 features (`EBM_FEATURES` and the PR 39 record), the three pairs
(`define_ebm_interactions`), the protocol (`EBMStage3Config`), and the hyperparameters, transcribed from glass_cascade
Cell 18's printed output and read from that notebook's pinned Git blob.

**Accepted artifact views, before any fit.** `SS.accepted_views` reads the accepted Stage 1 LR, GLASS router and EBM
out-of-fold outputs through the PR 38 boundary extractors at the PR 39 pins. An artifact whose file is absent is
reported as not available locally (the sandbox: complementarity is then not computed). An artifact that is present
but fails its pin, its boundary extraction or the training rows' labels stops the run here, naming the role and the
reason. For the acceptance run, set `PR42_REQUIRE_ACCEPTED_VIEWS=1`: all three views must then pass, and Cell 57 must
compute and reconcile the all-rows and Stage 2 abstention tables, so a completed run guarantees both.

In [ ]:
# ============================================================
# CELL 49: STEP 5A REGISTRATION AND THE BASELINE'S SOURCES
# ============================================================
REG_5A = SS.registration_checks()
print(SS.require(REG_5A, "Step 5A registration and baseline sources"))
print(f"{SS.REGISTRATION['id']} {SS.REGISTRATION['version']}  sha256 {SS.registration_digest()}")
ACCEPTED_EBM = SS.accepted_figures()
REQUIRE_VIEWS_5A = SS.views_required()
VIEWS_5A = SS.accepted_views(TRAIN.train_reference, require=REQUIRE_VIEWS_5A, y=TRAIN.y_train.to_numpy())
print(f"accepted artifact views (required: {REQUIRE_VIEWS_5A}): {VIEWS_5A['status']}")
print(f"hyperparameters {SS.PARAMS} ({SS.params_transcription()['source']})")
REG_5A

## Cell 50 — The training-side Stage 3 block and the past-only density
`SS.training_block` calls `Stage3Block.build` on the training rows with the production `EBMFeaturePipeline`, refitted
inside each of the five outer folds and their 5 × 5 inner folds (the builder needs a test frame: a one-row
placeholder from the training rows stands in and is never scored). Its columns are the 13 features, plus
`joint_economic_decay` and `neighborhood_subscription_density`, which must multiply to `decay_x_density` in every
frame. The **past-only density** replaces the density's reference rows with the rows of the same context contacted on
earlier days: the same kernel and scaler, refitted for each contact day on that history (Step 3B's construction).
The checks verify the values against the production `IntegralFeatureEngineer` fitted on exactly the available
rows, flip every held-out label of each fold, and flip the labels of rows contacted on or after a day, each with a
control that must move the values.

**Raw density and fallback.** The final feature, `decay_x_density_past` = `joint_economic_decay` × the past-only
density, reaches the EBM in two forms. A row with a raw past-only density (an earlier-day history) gets a value that
uses labels of earlier contacts only, which is feasible at prediction time under Step 3B's lag-0 assumption (the
outcome time is not recorded). A row without one (the first three contact days, or a history under 500 rows or 10
positives) gets the production cleaner's fallback: the median of `decay_x_density_past` over its fold's whole fit
side, which uses labels of contacts up to the fold's last day. `SS.fallback_trace` verifies both forms in the frames
the EBM uses, counts the held-out rows that get the fallback, and flips fit-side labels contacted after the last such
row's day (a tenth of them, then all): the fallback moves, while the first valued day's raw values (the control) do
not. The registered candidate is evaluated as fitted; its fallback rows are not past-only.

In [ ]:
# ============================================================
# CELL 50: STAGE 3 BLOCK AND THE PAST-ONLY DENSITY
# ============================================================
BLOCK_5A = SS.training_block(TRAIN.X_train, TRAIN.y_train)
BLOCK_CHECKS_5A = SS.block_checks(BLOCK_5A, TRAIN.X_train, TRAIN.y_train)
print(SS.require(BLOCK_CHECKS_5A, "training-side Stage 3 block"))
PAST_5A = SS.add_past_density(BLOCK_5A, TRAIN.X_train, TRAIN.y_train, TIME_INDEX)
PAST_CHECKS_5A = SS.past_density_checks(BLOCK_5A, TRAIN.X_train, TRAIN.y_train, TIME_INDEX, PAST_5A)
print(SS.require(PAST_CHECKS_5A, "past-only density"))
print(PAST_CHECKS_5A[["check", "passed", "detail"]].to_string(index=False))
FALLBACK_5A = SS.fallback_trace(BLOCK_5A, TRAIN.X_train, TRAIN.y_train, TIME_INDEX, PAST_5A)
FALLBACK_CHECKS_5A = SS.fallback_checks(FALLBACK_5A, PAST_5A)
print(SS.require(FALLBACK_CHECKS_5A, "past-only fallback trace"))
print(FALLBACK_5A["per_fold"].to_string(index=False))
print(FALLBACK_5A["sensitivity"].to_string(index=False))
print(FALLBACK_5A["totals"])
PAST_5A["contexts"]

## Cell 51 — The inventory, the correlated pair and the candidates
Distributions of every feature on the full-train block, and |Pearson r| among the 13 on the full-train block and on
each outer fit side. The registered rule requires `euribor3m_sigmoid_slope` / `emp_var_rate_sigmoid_slope` (the
frozen inventory's pruning note) to be the only pair at or above 0.7. `SS.configurations` resolves each candidate's
pairs by name with `define_ebm_interactions`: removing a feature loses its pair, replacing `decay_x_density` loses
`prior_x_stress × decay_x_density`, and a lost slot stays empty. Identical configurations would be fitted once.

In [ ]:
# ============================================================
# CELL 51: INVENTORY REVIEW, THE CORRELATED PAIR AND THE CANDIDATES
# ============================================================
DIST_5A = SS.distributions(PAST_5A["X_all"])
CORR_5A = SS.correlations(BLOCK_5A, PAST_5A["X_all"])
CORR_CHECKS_5A = SS.correlation_checks(CORR_5A)
print(SS.require(CORR_CHECKS_5A, "the remaining correlated pair"))
CONFIGS_5A = SS.configurations()
CONFIG_CHECKS_5A = SS.configuration_checks(CONFIGS_5A)
print(SS.require(CONFIG_CHECKS_5A, "pairs resolved by name"))
print(pd.DataFrame([{"configuration": k, "features": v["n_features"], "pairs": v["n_pairs"],
                     "lost": [SS._pair_name(p) for p in v["lost_pairs"]], "questions": v["questions"]}
                    for k, v in CONFIGS_5A.items() if "same_as" not in v]).to_string(index=False))
print({k: round(v, 4) for k, v in CORR_5A["extra"].items()})
CORR_5A["pairs"].head(8).round(4)

## Cell 52 — Fits, isolation and baseline parity
`SS.fit_all` runs every distinct configuration through the same protocol: five outer and 25 inner fits
(`OOFGenerator`), the fold-nested isotonic map (`Stage3Calibrator`), the global raw, global calibrated and
fold-nested thresholds (`F2ThresholdSelector`), and one all-rows fit used only for the shapes. Every fitted EBM's
terms must be its features plus its pairs, by name. With the fitted scores fixed, flipping each fold's held-out
labels must leave that fold's calibrated values and nested threshold unchanged (control: other folds move).
**Parity:** the refitted baseline against the PR 39 records, and row by row against the accepted EBM wherever its
view passed in Cell 49. Every candidate is compared with the refitted baseline, not with the artifact.
**The fallback's reach:** each of the past-only candidate's fold models, held fixed, is scored at its fold's fallback
and at the perturbed fallbacks of Cell 50; the change is what that fold's held-out fallback rows' term score would
move by.

In [ ]:
# ============================================================
# CELL 52: FITS, ISOLATION AND BASELINE PARITY
# ============================================================
FRAMES_5A = SS.frames_for_fits(BLOCK_5A, PAST_5A)
FITS_5A = SS.fit_all(CONFIGS_5A, BLOCK_5A, FRAMES_5A, verbose=True)
FIT_CHECKS_5A = pd.concat([SS.fit_checks(CONFIGS_5A, FITS_5A), SS.isolation_checks(FITS_5A["baseline"], BLOCK_5A)],
                          ignore_index=True)
print(SS.require(FIT_CHECKS_5A, "fitted terms, coverage and isolation"))
FALLBACK_5A["term_effect"] = SS.fallback_term_effect(FITS_5A["past_density"], CONFIGS_5A["past_density"], FRAMES_5A,
                                                     FALLBACK_5A)
print(FALLBACK_5A["term_effect"].round(4).to_string(index=False))
PARITY_5A = SS.parity(FITS_5A["baseline"], ACCEPTED_EBM, VIEWS_5A)
PARITY_5A

## Cell 53 — Paired out-of-fold comparison
For each configuration, `Stage3Metrics` at the three operating points, labelled raw or calibrated: the fold-nested
raw thresholds (primary), the global raw F2 threshold and the global calibrated threshold (both chosen on the rows
they are applied to). Then every candidate against the baseline on the same 32,950 rows: ROC-AUC with DeLong's
paired standard error, PR-AUC, Brier and ECE (10 equal-width bins) on raw scores and on calibrated probabilities,
nested F2, each per fold with the paired fold differences, and the nested decisions that change. Each difference is
the whole simplification's: a removed or replaced feature together with its lost pair.

In [ ]:
# ============================================================
# CELL 53: PAIRED OUT-OF-FOLD COMPARISON
# ============================================================
OPS_5A = SS.operating_point_table(FITS_5A, TRAIN.y_train)
DIFF_5A = SS.paired_differences(FITS_5A, TRAIN.y_train, BLOCK_5A.tune_plan.fold_id)
print(OPS_5A[["configuration", "operating_point", "probabilities", "threshold", "f2", "recall", "precision",
              "roc_auc", "pr_auc", "brier", "ece"]].round(4).to_string(index=False))
print(DIFF_5A["fold_differences"][["configuration", "metric", "mean", "sd", "min", "max", "folds_better"]]
      .round(4).to_string(index=False))
print(DIFF_5A["decisions"].to_string(index=False))
DIFF_5A["pooled"][["configuration", "roc_auc", "d_roc_auc", "d_roc_auc_low", "d_roc_auc_high", "pr_auc", "d_pr_auc",
                   "brier_calibrated", "d_brier_calibrated", "ece_calibrated", "ece_raw", "nested_f2",
                   "d_nested_f2"]].round(4)

## Cell 54 — Contributions across folds and contact periods
For the baseline: each term's importance (mean absolute term score) in the five fold models and the all-rows fit,
with its fold-to-fold variation, and each term's out-of-fold mean absolute contribution by Commit 3 contact period,
relative to its overall mean (each held-out row scored by its own fold's model). A term whose contribution is
concentrated in a few periods is carried by them.

In [ ]:
# ============================================================
# CELL 54: CONTRIBUTIONS ACROSS FOLDS AND CONTACT PERIODS
# ============================================================
REVIEW_5A = SS.contribution_review(FITS_5A, FRAMES_5A, CONFIGS_5A, TIME_INDEX)
print(REVIEW_5A["importance_summary"].round(4).to_string(index=False))
SS.show(SS.period_figure(REVIEW_5A))
DIST_5A.round(4)

## Cell 55 — Main-effect shapes, folds, support and periods
One figure per feature, one column per configuration that has it, on matched axes. The vertical axis is the EBM's
additive term score in log-odds, not a probability (the calibrated probability is a separate, monotone map of the
summed score). Thin lines are the five fold models, the thick line the all-rows fit, each evaluated on a common grid
with its own bins and shifted so its mean over the training rows is 0 (the EBM centres each term on its own fit rows;
the shift is recorded). Grey bands are sparse bins (under 1% of the rows). Below, the training rows per bin, stacked
by contact year, with bins that a single contact period holds more than half of hatched: a pattern there is evidence
only for that period. The decay-slot figure puts `decay_x_density` and its two replacements side by side.

**Post-registration centring check.** The fold curves of the two correlated slopes look vertically offset.
`SS.centring_check` re-centres each fold curve over well-supported values only and takes two centring-free
quantities per fold: the amplitude over those values and the regime step (mean contribution of the 2009–10 rows
minus that of the 2008 rows), also for the two slopes together, for every period-level main term together (a
feature is period-level when it is constant within every contact day up to floating-point noise; the table gives each
feature's largest within-day range), for those terms without the decay slot, and for the whole additive score. It
changes no fit, metric or ranking. The last figure plots each fold's regime step.
The notes below give a short reading of each changed feature.

In [ ]:
# ============================================================
# CELL 55: MAIN-EFFECT SHAPES ACROSS CONFIGURATIONS
# ============================================================
CURVES_5A = SS.shape_curves(FITS_5A, CONFIGS_5A, FRAMES_5A)
SUPPORT_5A = SS.support_tables(PAST_5A["X_all"], TIME_INDEX)
SHAPES_5A = SS.shape_summary(CURVES_5A, SUPPORT_5A)
for feature in SS.REVIEW_FEATURES:
    SS.show(SS.feature_figure(feature, CURVES_5A, SUPPORT_5A))
SS.show(SS.decay_slot_figure(CURVES_5A, SUPPORT_5A))
print(SHAPES_5A.round(4).to_string(index=False))

CENTRING_5A = SS.centring_check(FITS_5A, CONFIGS_5A, FRAMES_5A, SUPPORT_5A, TIME_INDEX)
print(CENTRING_5A["description"])
print("period-level features:", ", ".join(CENTRING_5A["period_level_features"]))
print(CENTRING_5A["within_day_spread"].to_string(index=False))
print(CENTRING_5A["centring"].drop(columns=["amplitude_by_fold", "step_by_fold"]).round(4).to_string(index=False))
print(CENTRING_5A["step_spread"].round(4).to_string(index=False))
print(CENTRING_5A["trade_off"].round(4).to_string(index=False))
SS.show(SS.regime_step_figure(CENTRING_5A))
for name, note in SS.FEATURE_NOTES.items():
    if " x " not in name:
        print(f"{name}: {note}\n")
CENTRING_5A["steps"].round(3)

## Cell 56 — Retained interactions
Each accepted pair in every configuration that keeps it, on support-defined cells of its two features (every
distinct value, or the distinct 5% quantiles), the same cells in every panel. A cell's value is the mean pair-term
contribution of its training rows (log-odds, centred to mean 0 over the training rows), so nothing is drawn outside
the joint support: hatched cells hold no rows and x marks sparse cells (under 0.1% of the rows). Top row: joint
support, then each model's all-rows score on one symmetric scale per pair. Bottom row: the largest contact period's
share per cell, then each model's per-cell sd across its five fold models on one shared scale. (The registered
regular grid left almost every cell empty, so it was replaced for display after inspection; the scores are the same
models'.) The table gives each surface's largest and full range over well-supported cells and its fold spread.

In [ ]:
# ============================================================
# CELL 56: RETAINED INTERACTIONS
# ============================================================
PAIRS_5A = SS.pair_surfaces(FITS_5A, CONFIGS_5A, FRAMES_5A, TIME_INDEX)
for pair in [f"{a} & {b}" for a, b in SS.BASELINE_PAIRS]:
    SS.show(SS.pair_figure(pair, PAIRS_5A["surfaces"]))
for name, note in SS.FEATURE_NOTES.items():
    if " x " in name:
        print(f"{name}: {note}\n")
SS.pair_table(PAIRS_5A).round(4)

## Cell 57 — Complementarity with Stage 1, and the nominations
Where the accepted Stage 1 LR, GLASS router and EBM views all passed (Cell 49), every EBM is aligned with
Stage 1 by row identity: Stage 1 decisions at `lr.in_stage_f2_cv`, the accepted EBM's stored `ebm.nested_f2`
decisions, each refitted EBM's decisions at its own fold-nested thresholds, and Stage 2 abstentions from the router's
out-of-fold routes. For all rows and for abstentions: both correct, Stage 1 only correct, EBM only correct, both
wrong, reconciled to the population; then each candidate against the refitted baseline (Stage 1 errors corrected,
correct Stage 1 decisions overturned, net). A disagreement count does not establish a better cascade decision.
`SS.complementarity_requirement` then holds the acceptance run to it: with `PR42_REQUIRE_ACCEPTED_VIEWS=1`, the run
stops unless complementarity was computed for the accepted EBM and every configuration, each with a reconciled
all-rows and Stage 2 abstention table. `SS.nominations` applies the registered Step 5B and cascade rules.

In [ ]:
# ============================================================
# CELL 57: COMPLEMENTARITY WITH STAGE 1 AND THE NOMINATIONS
# ============================================================
if VIEWS_5A["available"]:
    DECISIONS_5A = {"accepted": VIEWS_5A["ebm"]["decision"],
                    **{k: (r["proba"] >= r["nested"]["per_row"]).astype(int) for k, r in FITS_5A.items()}}
    COMP_5A = SS.complementarity(TRAIN.y_train.to_numpy(), VIEWS_5A["lr"]["flags"], VIEWS_5A["router"]["abstain"],
                                 DECISIONS_5A)
    COMP_CHECKS_5A = SS.complementarity_checks(COMP_5A, len(TRAIN.y_train))
    print(SS.require(COMP_CHECKS_5A, "complementarity tables"))
    print(COMP_5A["table"].to_string(index=False))
    print(COMP_5A["against_reference"].to_string(index=False))
else:
    COMP_5A, COMP_CHECKS_5A = None, pd.DataFrame(columns=["check", "passed", "detail"])
    print(f"complementarity not computed: accepted artifacts {VIEWS_5A['status']}")
COMP_REQUIRED_5A = SS.complementarity_requirement(VIEWS_5A, COMP_5A, COMP_CHECKS_5A, ["accepted", *FITS_5A])
print(SS.require(COMP_REQUIRED_5A, "complementarity requirement"))
NOMS_5A = SS.nominations(DIFF_5A, COMP_5A)
for name, judgement in SS.INTERPRETATION.items():
    print(f"{name} (interpretability gain: {judgement['gain']}): {judgement['rationale']}\n")
NOMS_5A

## Cell 58 — The review document's statements
`SS.stated_checks` requires every figure the review document states to hold in this run: structure and
model-independent counts exactly, model-dependent values within the tolerance stated beside each (the accepted
Windows run can differ from the sandbox in the last digits), and the nominations.

In [ ]:
# ============================================================
# CELL 58: THE REVIEW DOCUMENT'S STATEMENTS
# ============================================================
STATED_5A = SS.stated_checks(configs=CONFIGS_5A, corr=CORR_5A, past=PAST_5A, diff=DIFF_5A, mtable=OPS_5A,
                             summary=SHAPES_5A, noms=NOMS_5A, centring=CENTRING_5A, pairs=PAIRS_5A,
                             fallback=FALLBACK_5A, comp=COMP_5A)
print(SS.require(STATED_5A, "the review document's statements"))
STATED_5A

## Cell 59 — Research record: Stage 3 simplification
The run's sixth record: `research_logs/pr42/<run_id>/stage3_simplification.json` and its `.sha256`, read back through
both. It holds the registration and its checks, the row and fold identities, the past-only contexts, checks and
fallback trace, the review, the configurations and fitted terms, parity, the paired comparison, the shapes, support
and interactions, the post-registration centring check, complementarity (or its unavailability) with the views'
requirement, the nominations, the interpretation and feature notes, and the limits.

In [ ]:
# ============================================================
# CELL 59: WRITE THE STAGE 3 SIMPLIFICATION RECORD
# ============================================================
SIMPLIFICATION_PATH = RR.write_run_record(RUN, "stage3_simplification", SS.record_sections(
    reg_checks=REG_5A, block=BLOCK_5A, X=TRAIN.X_train, block_chk=BLOCK_CHECKS_5A, past=PAST_5A,
    past_chk=PAST_CHECKS_5A, dists=DIST_5A, corr=CORR_5A, corr_chk=CORR_CHECKS_5A, configs=CONFIGS_5A,
    cfg_chk=CONFIG_CHECKS_5A, fits=FITS_5A, fit_chk=FIT_CHECKS_5A, parity_table=PARITY_5A, mtable=OPS_5A,
    diff=DIFF_5A, review=REVIEW_5A, curves=CURVES_5A, supports=SUPPORT_5A, summary=SHAPES_5A, pairs=PAIRS_5A,
    views_status={"status": VIEWS_5A["status"], "required": VIEWS_5A["required"]}, comp=COMP_5A,
    comp_chk=pd.concat([COMP_CHECKS_5A, COMP_REQUIRED_5A], ignore_index=True), noms=NOMS_5A, stated=STATED_5A,
    centring=CENTRING_5A, fallback=FALLBACK_5A, fallback_chk=FALLBACK_CHECKS_5A))
SIMPLIFICATION_RECORD = RR.read_run_record(RUN.run_id, "stage3_simplification")
print(f"record  : {SIMPLIFICATION_PATH.relative_to(BANK_ROOT)}")
print(f"content : {SIMPLIFICATION_RECORD['content_sha256']}")

## Cell 60 — Step 5B: registration and the history of the accepted pairs
**Question.** On the frozen 13 features, does a reproducible selection procedure recover PR 39's three accepted pairs
on all training rows, and which pairs recur when it is repeated inside each outer fold? For `jed_alone` and
`drop_first`, what does reselection under the same three-pair budget recover, and how does each reselected fit
compare with its own Step 5A fixed-pair fit on identical held-out rows?

**The selector** is interpret's own FAST ranking inside the accepted Stage 3 EBM: `interactions=3` in place of an
explicit pair list. In each of the EBM's 14 outer bags, after the main effects are boosted, every pair of the fit's
features is ranked by FAST strength on that bag's training rows. The three pairs with the smallest mean rank are
boosted, ties broken by column order. Every eligible pair is ranked, including `campaign × emp_var_rate_sigmoid_slope`.
A recorder observes the per-bag rankings without changing the fit, and its top three must equal the fitted pairs.

`IS.REGISTRATION` (1.0.0) fixes the selector, the eligible pairs, the ranking, ties, the budget and every comparison,
and was written before any Step 5B selection ran. This cell checks its digest and reads back the history it cites:
- `glass_pipeline/ebm/interactions.py` labels one TIER 1 and two TIER 2 ("exploratory") pairs, and its TIER 1
  comment gives "importance=0.205".
- The exploratory notebook is read as Step 4A pinned it (`BD.EXPLORATORY`: Git blob `266860ba…`, LF SHA-256
  `136c8fb7…`), never from the working copy, so clearing that copy's outputs changes nothing here. Its Cells 16A/16B,
  found by their header lines, hold the research round: interpret's FAST with `interactions 4`, and its four pairs
  with their importances.
- "0.205" appears in none of that pinned notebook's printed outputs (every code cell with output, Cells 16A and 16B
  among them; Cell 16B prints 0.0529 for the TIER 1 pair). The statement covers those outputs only: other notebooks,
  logs and commits were not inspected. The registration's "no tracked output gives it" is broader than what was
  inspected; this is the form the evidence supports.

The source's identity (commit, blob, digest, and whether Git or the working copy supplied it) goes into the record.
If Git lacks the blob and the working copy is not the pinned notebook, the history is reported as not checked; a
mismatch stops the run.

**The accepted Step 5A record.** By Step 3B's rule, this run's Step 5A record (Cell 59) is compared with the accepted
Windows run `c5_20261124_002645` (content `d70330e1…`): its integrity wherever its directory exists, and exact
reproduction where this run has that run's environment and views. The acceptance run (`PR42_REQUIRE_ACCEPTED_VIEWS=1`)
needs the directory. The check runs here, before any Step 5B fit.

In [ ]:
# ============================================================
# CELL 60: STEP 5B REGISTRATION, THE HISTORY OF THE ACCEPTED PAIRS AND THE ACCEPTED STEP 5A RECORD
# ============================================================
HISTORY_5B = IS.historical_provenance()
REG_5B = IS.registration_checks(history=HISTORY_5B)
print(SS.require(REG_5B, "Step 5B registration and history"))
print(REG_5B[["check", "passed"]].to_string(index=False))
print("history source:", {k: HISTORY_5B["source"][k] for k in ("commit", "blob", "sha256_lf", "source", "status")})
ACCEPTED_5A_5B = IS.accepted_5a_reproduction(SIMPLIFICATION_RECORD, RUN.environment, VIEWS_5A)
ACCEPTED_5A_CHECKS_5B = IS.accepted_5a_checks(ACCEPTED_5A_5B)
print(ACCEPTED_5A_5B.drop(columns=["accepted_content_sha256"]).to_string(index=False))
print(SS.require(ACCEPTED_5A_CHECKS_5B, "accepted Step 5A record (integrity; reproduction where required)"))
pd.Series(IS.REGISTRATION["selector"])

## Cell 61 — Recovery on the frozen 13 features
The selector runs inside each outer fold's 26,360 fit rows (through `OOFGenerator`, as every configuration is fitted)
and once on all 32,950 training rows. The all-rows selection is descriptive: it is the counterpart of PR 39's
research round on today's features and settings. The outer-fold selections are the evidence of recurrence. Each
selection is matched to the fit frame whose rows it used.

In [ ]:
# ============================================================
# CELL 61: RECOVERY ON THE FROZEN 13 FEATURES
# ============================================================
REC_5B = IS.recovery(BLOCK_5A, FRAMES_5A)
REC_CHECKS_5B = IS.selection_checks("frozen 13 features", REC_5B["selections"], len(REC_5B["features"]),
                                    expect={"outer": 5, "inner": 0, "all rows": 1})
print(SS.require(REC_CHECKS_5B, "recovery selections"))
REC_TABLE_5B = IS.recurrence(REC_5B["selections"], IS.ACCEPTED_PAIRS)
print(IS.selected_by_fit(REC_5B["selections"]).to_string(index=False))
print(IS.recovery_summary(REC_5B, REC_TABLE_5B))
REC_TABLE_5B.head(12).round(3)

## Cell 62 — The candidates, reselected
`jed_alone` and `drop_first` go through Step 5A's protocol unchanged (same features, rows, folds, seeds, settings,
calibration and operating points), except that every fit selects its own three pairs: the 5 outer fits, the 25 inner
fits whose scores feed calibration and the fold-nested thresholds, and the all-rows fit for the surfaces. The checks
are Step 5A's (fitted terms, coverage, fold isolation of calibration and thresholds) plus the selections' own: each on
exactly one fit frame's rows, all eligible pairs ranked in all 14 bags, the recorded top three equal to the fitted
pairs.

In [ ]:
# ============================================================
# CELL 62: THE CANDIDATES, RESELECTED
# ============================================================
CONFIGS_5B = IS.configurations(CONFIGS_5A)
FITS_5B = {name: IS.fit_selected(c, BLOCK_5A, FRAMES_5A) for name, c in CONFIGS_5B.items()}
FIT_CHECKS_5B = pd.concat(
    [SS.fit_checks(CONFIGS_5B, FITS_5B)] + [SS.isolation_checks(r, BLOCK_5A) for r in FITS_5B.values()]
    + [IS.selection_checks(n, r["selections"], len(CONFIGS_5B[n]["features"]),
                           expect={"outer": 5, "inner": 25, "all rows": 1}) for n, r in FITS_5B.items()],
    ignore_index=True)
print(SS.require(FIT_CHECKS_5B, "reselected fits, isolation and selections"))
for name, r in FITS_5B.items():
    print(f"{name} (5A pairs: {', '.join(IS.pair_label(a, b) for a, b in CONFIGS_5B[name]['five_a_pairs'])})")
    print(IS.selected_by_fit(r["selections"]).to_string(index=False))

## Cell 63 — Recurrence, and what enters, persists or disappears
For each configuration, every eligible pair's outer-fold selection count (stable 5 of 5, recurring at least 3,
occasional 1–2, never), its inner-fold count (supporting), whether the all-rows fit selected it, and its mean FAST rank
across the outer fits. Against the reference pairs (the accepted three for the frozen features; each candidate's two
5A pairs), a reference pair persists if at least three outer fits select it and disappears otherwise, and a new pair
enters if at least three do. The figure shows each outer fit's and the all-rows fit's selections.

In [ ]:
# ============================================================
# CELL 63: RECURRENCE, AND WHAT ENTERS, PERSISTS OR DISAPPEARS
# ============================================================
TABLES_5B = {"baseline": REC_TABLE_5B, **{n: IS.recurrence(r["selections"], CONFIGS_5B[n]["five_a_pairs"])
                                          for n, r in FITS_5B.items()}}
SELECTIONS_5B = {"baseline": REC_5B["selections"], **{n: r["selections"] for n, r in FITS_5B.items()}}
SS.show(IS.recurrence_figure(TABLES_5B, SELECTIONS_5B))
for name, t in TABLES_5B.items():
    print(name)
    print(t[(t["outer_selected"] > 0) | t["reference"] | t["all_rows_selected"]].round(2).to_string(index=False))
IS.change_summary(TABLES_5B)

## Cell 64 — Each reselected candidate against its own 5A fit
On the same 32,950 out-of-fold rows: pooled ROC-AUC (DeLong's paired difference and 95% CI), PR-AUC, nested F2, Brier
and ECE (raw and calibrated, 10 bins), per-fold differences, and changed decisions. The 5A fits hold two pairs and the
reselected fits three, so a difference belongs to the whole procedure (which pairs and how many). The registered
descriptive thresholds mark a material change; they are not an adoption rule.

In [ ]:
# ============================================================
# CELL 64: EACH RESELECTED CANDIDATE AGAINST ITS OWN 5A FIT
# ============================================================
FOLD_ID_5B = BLOCK_5A.tune_plan.fold_id
VS_5A = {n: IS.paired_against(FITS_5A[c["five_a"]], FITS_5B[n], TRAIN.y_train, FOLD_ID_5B)
         for n, c in CONFIGS_5B.items()}
CMP_5B = IS.comparison_table(VS_5A)
for name, c in VS_5A.items():
    print(name)
    print(c["fold_differences"][["metric", "mean", "sd", "min", "max", "folds_better"]].round(4).to_string(index=False))
CMP_5B.set_index("configuration").T

## Cell 65 — Learned surfaces of the recurring and changed pairs
Every pair that recurs (selected by at least three outer fits) in any configuration, and every 5A pair that
disappears, on Step 5A's support-defined cells. Each figure compares the models that hold the pair in their all-rows
fit on matched cells and one symmetric colour scale. The fold sd uses the outer-fold models that selected the pair; with fewer than two selecting folds it does not exist, and figure and table say "unavailable" (never 0). The table gives the selecting folds as "n of 5".
The table gives what each pair adds over its features' main effects (mean |contribution| of the pair and of each main
effect, the pair's share of all terms), its largest well-supported effect, and how often the fold models agree with
the all-rows sign where the effect is at least 0.05 log-odds.

In [ ]:
# ============================================================
# CELL 65: LEARNED SURFACES OF THE RECURRING AND CHANGED PAIRS
# ============================================================
SHOW_5B = IS.pairs_to_show(TABLES_5B)
MODELS_5B = IS.model_sets(FITS_5A, CONFIGS_5A, FITS_5B, CONFIGS_5B)
SURF_5B = IS.surfaces(MODELS_5B, FRAMES_5A["X_all"], TIME_INDEX, SHOW_5B)
for pair in dict.fromkeys(s["pair"] for s in SURF_5B):
    SS.show(SS.pair_figure(pair, SURF_5B, order=IS.ORDER, labels=IS.LABELS))
VALUE_5B = IS.interaction_value(MODELS_5B, FRAMES_5A["X_all"], SURF_5B)
print("shown:", ", ".join(SHOW_5B))
IS.value_summary(VALUE_5B)

## Cell 66 — Complementarity under Step 5A's cascade screen
Where the accepted views passed (Cell 49), the four paired counts and the net corrections of Stage 1 against the 5A
baseline are computed for all rows and Stage 2 abstentions, with the reselected fits beside their 5A fits. Step 5A's
screen is applied unchanged to each: within the 5A tolerances against the baseline, and an abstention net of at least
the baseline's net minus 0.5% of the abstention rows. In the accepted 5A run `jed_alone` met it (486) and `drop_first`
missed it (441, needed 456.86). A changed verdict is reported as such, not as a better cascade. With
`PR42_REQUIRE_ACCEPTED_VIEWS=1` the run stops unless these tables were computed and reconciled.

In [ ]:
# ============================================================
# CELL 66: COMPLEMENTARITY UNDER STEP 5A'S CASCADE SCREEN
# ============================================================
SCREEN_5B = IS.cascade_screen(VIEWS_5A, FITS_5A, FITS_5B, TRAIN.y_train, FOLD_ID_5B)
EBMS_5B = ["accepted", "baseline", *IS.CANDIDATES, *FITS_5B]
if SCREEN_5B is None:
    print(f"complementarity not computed: accepted artifacts {VIEWS_5A['status']}")
    SCREEN_REQUIRED_5B = SS.complementarity_requirement(VIEWS_5A, None, None, EBMS_5B)
else:
    print(SS.require(SCREEN_5B["checks"], "complementarity tables (5B)"))
    print(SCREEN_5B["comp"]["table"].to_string(index=False))
    print(SCREEN_5B["comp"]["against_reference"].to_string(index=False))
    print(SCREEN_5B["screen"].to_string(index=False))
    print(SCREEN_5B["change"].to_string(index=False))
    SCREEN_REQUIRED_5B = SS.complementarity_requirement(VIEWS_5A, SCREEN_5B["comp"], SCREEN_5B["checks"], EBMS_5B)
print(SS.require(SCREEN_REQUIRED_5B, "complementarity requirement (5B)"))

## Cell 67 — Acceptance, and reproduction of the sandbox run
Two different questions, kept apart.

**Acceptance** (`IS.acceptance_checks`) must hold in every run:
- fit isolation: Step 5A's checks for every reselected fit, and every selection on its own fit frame's rows (each
  outer selection on exactly the rows outside its held-out fold);
- ranking-to-fitted-term agreement: each selection's recorded top three are the fitted pairs, every eligible pair is
  ranked in all 14 bags, and the models that scored the held-out rows hold exactly those pairs;
- paired row identity between each reselected fit and its 5A fit;
- the accepted Step 5A reproduction (Cells 58 and 60);
- truthful verdicts: the registered thresholds are intact, and each materiality and cascade verdict reported is
  recomputed here from this run's numbers.

**Same-sandbox reproduction** (`IS.sandbox_reproduction`) compares this run with the sandbox run that the review
document quotes (`IS.SANDBOX_REFERENCE`, read from that run's record). It covers every selection with its margin, every
pair's counts and positions, and every estimate, difference and verdict, each with the sandbox value, this run's value
and the difference. It also evaluates the review document's statements (`IS.STATED`).

Exact reproduction is required only in the sandbox's environment. Elsewhere, including the Windows acceptance run,
nothing in it fails and the differences are recorded. These are flagged for review:
- a changed selection or count;
- a change of direction in a registered comparison, below the materiality threshold or not;
- a changed verdict;
- a statement that no longer holds;
- a cascade verdict that reselection changes.

The registered thresholds are not adjusted.

In [ ]:
# ============================================================
# CELL 67: ACCEPTANCE, AND REPRODUCTION OF THE SANDBOX RUN
# ============================================================
STATED_5B = IS.stated_checks(recovery=REC_5B, tables=TABLES_5B, fits=FITS_5B, cmp=CMP_5B, vs=VS_5A, value=VALUE_5B,
                             screen=SCREEN_5B)
SECTIONS_5B = IS.record_sections(
    reg_checks=REG_5B, history=HISTORY_5B, accepted_5a=ACCEPTED_5A_5B, accepted_5a_checks=ACCEPTED_5A_CHECKS_5B,
    recovery=REC_5B, rec_checks=REC_CHECKS_5B, configs=CONFIGS_5B, fits=FITS_5B, fit_checks=FIT_CHECKS_5B,
    tables=TABLES_5B, vs=VS_5A, cmp=CMP_5B, shown=SHOW_5B, surfaces=SURF_5B, value=VALUE_5B, views=VIEWS_5A,
    screen=SCREEN_5B, screen_required=SCREEN_REQUIRED_5B)
ACCEPT_5B = IS.acceptance_checks(recovery=REC_5B, fits_5a=FITS_5A, fits_5b=FITS_5B, configs_5b=CONFIGS_5B,
                                 fit_checks=FIT_CHECKS_5B, frames=FRAMES_5A, y=TRAIN.y_train, fold_id=FOLD_ID_5B,
                                 cmp=CMP_5B, screen=SCREEN_5B, noms_5a=NOMS_5A, stated_5a=STATED_5A,
                                 accepted_5a=ACCEPTED_5A_CHECKS_5B, views=VIEWS_5A)
REPRO_5B = IS.sandbox_reproduction(SECTIONS_5B, STATED_5B, step5a_content=SIMPLIFICATION_RECORD["content_sha256"],
                                   environment=RUN.environment, views=VIEWS_5A)
REVIEW_5B = IS.review_flags(REPRO_5B, SCREEN_5B)
print(ACCEPT_5B[["check", "passed"]].to_string(index=False))
print(f"\nagainst the sandbox run {REPRO_5B['reference']['run_id']} (required: {REPRO_5B['required']}; "
      f"{REPRO_5B['basis']})")
print(IS.reproduction_summary(REPRO_5B).to_string(index=False))
DIFFER_5B = REPRO_5B["table"][~REPRO_5B["table"]["equal"]]
print(DIFFER_5B.head(60).to_string(index=False) if len(DIFFER_5B) else "every item equals the sandbox run's")
print(f"\nflagged for review: {len(REVIEW_5B)}")
REVIEW_5B

## Cell 68 — Research record: interaction selection
The run's seventh record: `research_logs/pr42/<run_id>/interaction_selection.json` and its `.sha256`, read back
through both. It holds:
- the registration, its checks and the history with its pinned source;
- the accepted Step 5A comparison;
- every selection (fit, rows, ranking of every eligible pair);
- the recovery and recurrence tables, the reselected fits' checks, thresholds and out-of-fold digests;
- the comparison with Step 5A, the surfaces and interaction values, and complementarity (or its unavailability);
- the acceptance checks with their verdict, the sandbox reproduction (items, statements, checks) and everything
  flagged for review;
- the limits.

The record is written first. Acceptance, and in the sandbox's environment the reproduction, are enforced after it, so a
run that stops here still leaves its evidence to inspect without repeating the fits.

In [ ]:
# ============================================================
# CELL 68: WRITE THE INTERACTION SELECTION RECORD, THEN ENFORCE ACCEPTANCE
# ============================================================
SELECTION_PATH = RR.write_run_record(RUN, "interaction_selection",
                                     IS.final_sections(SECTIONS_5B, ACCEPT_5B, REPRO_5B, REVIEW_5B))
SELECTION_RECORD = RR.read_run_record(RUN.run_id, "interaction_selection")
print(f"record  : {SELECTION_PATH.relative_to(BANK_ROOT)}")
print(f"content : {SELECTION_RECORD['content_sha256']}")
print(SS.require(ACCEPT_5B, "Step 5B acceptance"))
print(SS.require(REPRO_5B["checks"], "same-sandbox reproduction (required only in the sandbox's environment)"))
print(f"flagged for review: {len(REVIEW_5B)}")

## PR 42 conclusion — feature-contract dispositions and handoff
Commit 6, Step 6A consolidates this notebook's accepted records in
`research_logs/pr42/feature_contract_recommendations.md`. It adds no cell and no record. The document lists the accepted
runs with their digests and an evidence index of the figures behind each disposition;
`tests/feature_research/test_pr42_contract_recommendations.py` resolves every index row in its record.

- **Every frozen contract is retained.** That covers the Stage 1 contract 2.0.0, the Stage 2 binning, the 13 Stage 3
  features and their DAG, and the three EBM pairs. PR 42 adopts nothing; adoption is PR 45's.
- **Advanced, each to its own later evaluation:**
  - the past-only Stage 1 encodings, drafted as `stage1-features` 3.0.0-candidate, and the MLP's encoder-isolated
    early stopping (Step 44A);
  - NSD `gap_placed`, qualified by the fold-local sensitivity, and `eci_warm` required as a filter on Pass 2 partner
    rules (Step 44B);
  - `jed_alone` with its two Step 5A pairs, and the procedural interaction selector with the budget question
    (Step 44C).
- **Not advanced.** Reselection under the three-pair budget is not advanced: in the accepted Step 5B run it makes
  `jed_alone` miss the Step 5A cascade screen (389 net corrections on Stage 2 abstentions, against 456.86).
- **Recommended as separately versioned methodology revisions:**
  - past-only encodings for any claim about prediction time;
  - encoder-isolated MLP early stopping;
  - fold-fitted derivation of label-chosen constants, with stated reference inputs, in nested and repeated
    evaluations.
- **Claim boundary.** The frozen training-side evidence is valid random-split evidence: no held-out label reaches its
  own encoding. It is not forward-in-time evidence: every target-derived feature uses labels of later contacts.
- **Stage 3 feature DAG.** One production-owned implementation under `shared/` should replace the Stage 2 copies.
  It belongs to the adopting contract version, with compatibility aliases for the class paths pickled in the PR 39
  Stage 3 artifacts. Nothing moves in PR 42.

Sections added after Commit 2, on the same training side and record conventions:

| Commit | Section | Sources |
|---|---|---|
| 3 | Step 3A (Cells 14–21): time index, encoding construction, fitting paths, fold isolation, attribution by contact timing, provenance, findings. Step 3B (Cells 22–30): past-only Stage 1 encodings, availability, invariance, predictive effect (encoding versus split, MLP early stopping), contract decisions, the PR 49 protocol and the interim finding | Cells 2, 6–8 here; `shared.stage1.crossfit`, `stage2_crossfit`, `shared.stage3.data` |
| 4 | Step 4A (Cells 31–39): the PR 39 NSD/ECI bin evidence reproduced by import, held-out label flips, macro states and the in-sample / fold-fit / out-of-fold encoder comparison, support and fold stability, decile lift and the historical thresholds, contact periods and straddling states, Commit 3 label provenance by bin, findings. Step 4B (Cells 40–48): the registration, the reference inputs, fold-fitted procedures and their cuts, held-out label flips for the fold fits, the comparison with the frozen bins on identical held-out rows, fold and period dependence, `eci_warm` as a filter on partner rules, the decision, a post-registration sensitivity for NSD `gap_placed` | E 13C–13D at `1ce1649`, E 15A–15B; `stage2_binning`, `stage2_crossfit`, `encoding_audit`; the GLASS primary profile and the shared contract |
| 5 | Step 5A (Cells 49–59): the registration, the training-side Stage 3 block and the past-only density, the inventory review and the correlated pair, the candidates and their pairs, fits with isolation and baseline parity, the paired out-of-fold comparison, contributions across folds and periods, every main-effect shape and retained interaction with support, complementarity with Stage 1 where the accepted artifacts exist, the nominations. Step 5B (Cells 60–68): the registration and the history of the accepted pairs, recovery on the frozen 13 features, the candidates reselected inside every fit, recurrence and change, the comparison with each 5A fit, the surfaces of recurring and changed pairs, complementarity under the 5A cascade screen, acceptance and reproduction of the sandbox run | E 7–10; G 7, G 17–18; `shared.stage3`, `glass_pipeline.ebm`, `encoding_audit`, `temporal_validity` |
| 6 | Step 6A (no cell): the feature-contract dispositions, the versioning and adoption requirements, the Stage 3 feature DAG decision and the handoff, in `research_logs/pr42/feature_contract_recommendations.md`. Steps 6B–6D: executed-notebook packaging, final validation and publication | the accepted records of Commits 2–5 |